# PepperLens FINAL — DSP / digital image processing laboratory

**Build: 24 October 2026.** Upload this one notebook to a **hosted Google Colab CPU runtime**, then run the cells in order or choose **Runtime → Run all**. No separate Python files and no training-image uploads are required. This file contains its source code and the dataset API links; the photographs are fetched directly into the cloud runtime.

The main aim is **digital signal and image processing applied to bell pepper leaves**. The visible cells below contain the actual cleaning, colour-conversion, smoothing, segmentation, feature-extraction and model-definition code used by training. A later laboratory section plots each intermediate image, Gaussian filter response and extracted feature values using a training example. The compact setup cell only packages the supporting files.

The default source reports **9,283 bell pepper leaf images** across healthy, bacterial spot, Cercospora leaf spot, leaf curl, nutrient deficiency and powdery mildew. Existing compatible data in the current cloud session are reused. Its actual usable size is measured after cleaning, not assumed to equal the publisher's file count.

RGB/HSV masking → colour, texture and shape features → Logistic Regression, LDA, KNN, Decision Tree, Random Forest, Gaussian Naive Bayes and SVM. **The main classifier is selected on validation balanced accuracy before the test set is evaluated.** Every model's prediction is shown; equal voting is retained only as an optional comparison.

Identical images with conflicting labels are reported and all copies are excluded. Source images are preserved. Missing dependencies, helper-file generation, stale imports, partial transfers and corrupt feature-cache entries are handled in the setup/workflow below. A data source that is unavailable or lacks sufficient correctly labelled examples still needs attention; the notebook will not invent labels, images or accuracy.

No six-class accuracy is prefilled. Local regression tests use generated fixtures; the live six-class Colab run has not been executed by the assistant. Unknown augmentation/plant lineage means the reported accuracy remains a provisional internal result until independently evaluated.

In [ ]:
# 1. Install only missing/incompatible dependencies, before importing the ML stack.
import importlib.metadata as metadata
import os
import subprocess
import sys

os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ['MPLBACKEND'] = 'Agg'
try:
    from packaging.specifiers import SpecifierSet
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'packaging'])
    from packaging.specifiers import SpecifierSet

DEPENDENCIES = {
    'numpy': '>=1.26,<3', 'opencv-python-headless': '>=4.9,<6',
    'mahotas': '>=1.4.13,<2', 'scikit-learn': '>=1.4,<2',
    'pandas': '>=2.1,<4', 'pillow': '>=10,<13', 'joblib': '>=1.3,<2',
    'gradio': '>=6,<7', 'ipywidgets': '>=8,<9', 'matplotlib': '>=3.8,<4',
    'kagglehub': '',
}
needed = []
for package, specifier in DEPENDENCIES.items():
    try:
        installed = metadata.version(package)
        compatible = not specifier or installed in SpecifierSet(specifier)
    except metadata.PackageNotFoundError:
        compatible = False
    if not compatible:
        needed.append(package + specifier)
if needed:
    print('Installing:', ', '.join(needed))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *needed])
print('Dependencies ready. Python:', sys.version.split()[0])

The next cell reconstructs the embedded helper files in a dedicated code folder and loads those exact files. You do not need to upload `cloud_data.py` or `pepper_model.py`, and there are no `from cloud_data ...` imports in the notebook cells for the editor to resolve. The source is available in the runtime's Files panel for inspection.

In [ ]:
# 2. Prepare the embedded source and verify imports.
SOURCES = {
  "cloud_data.py": "\"\"\"Cloud-only dataset acquisition. No network is used merely by importing this file.\"\"\"\nimport importlib.util\nimport json\nimport os\nimport shutil\nfrom pathlib import Path\n\nDATASET_OPTIONS = {\n    'mendeley_bell_6': {\n        'page': 'https://data.mendeley.com/datasets/w9drvtxg9g/1',\n        'reported_images': 9283, 'scope': 'Bell pepper; six condition classes including healthy',\n        'access': 'Mendeley archive; about 705 MB on the previously verified download page',\n    },\n    'plantifydr_pepper': {\n        'page': 'https://www.kaggle.com/datasets/lavaman151/plantifydr-dataset',\n        'kaggle_handle': 'lavaman151/plantifydr-dataset',\n        'reported_images': None,\n        'scope': 'Bell pepper subset of 125k+ multicrop files; augmented; subset count must be measured',\n    },\n    'agml_pepper_5': {\n        'page': 'https://huggingface.co/datasets/Project-AgML/pepper_disease_pest_classification',\n        'huggingface_id': 'Project-AgML/pepper_disease_pest_classification',\n        'reported_images': 26377,\n        'scope': 'General pepper, five disease labels, no healthy class; bell cultivar not confirmed',\n        'status': 'Review only: dataset-card taxonomy differs from the linked paper',\n    },\n}\n\ndef require_cloud_runtime(allow_local_download=False):\n    \"\"\"A guard against accidentally downloading large images to the user's PC.\"\"\"\n    in_colab = bool(os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_BACKEND_VERSION'))\n    try:\n        in_colab = in_colab or importlib.util.find_spec('google.colab') is not None\n    except (ModuleNotFoundError, ValueError):\n        pass\n    in_kaggle = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))\n    if not (in_colab or in_kaggle or allow_local_download):\n        raise RuntimeError('Dataset downloads are restricted to Colab/Kaggle for this project. '\n                           'Open the notebook in a cloud runtime. No images were downloaded.')\n\ndef import_plantifydr(root='/content/pepper_plantifydr'):\n    \"\"\"Download via Kaggle API to cloud disk, then index only Bell pepper files.\n\n    No credentials are written into source code. If Kaggle requests authentication,\n    use kagglehub.login() interactively inside Colab. The current KaggleHub API\n    can select this directory; update kagglehub if an old version rejects it.\n    \"\"\"\n    require_cloud_runtime()\n    import kagglehub\n    cached = Path(kagglehub.dataset_download('lavaman151/plantifydr-dataset',\n                  path='PlantDiseasesDataset/Bell pepper'))\n    candidates = [p for p in [cached, *cached.rglob('*')]\n                  if p.is_dir() and p.name.lower() == 'bell pepper']\n    if len(candidates) != 1:\n        raise ValueError(f'Expected one Bell pepper folder; found {len(candidates)}. Inspect {cached}.')\n    source = candidates[0]\n    root = Path(root)\n    mapping = {}\n    for folder in source.iterdir():\n        if folder.is_dir():\n            name = folder.name.lower().replace('_', ' ')\n            if 'bacterial' in name:\n                mapping[folder] = 'bacterial_spot'\n            elif 'healthy' in name.split():\n                mapping[folder] = 'healthy'\n            else:\n                raise ValueError(f'Unrecognized pepper label {folder.name}; do not guess its diagnosis.')\n    if set(mapping.values()) != {'healthy', 'bacterial_spot'}:\n        raise ValueError('Both healthy and bacterial spot classes must be present.')\n    counts = {'healthy': 0, 'bacterial_spot': 0}\n    for folder, label in mapping.items():\n        for file in folder.rglob('*'):\n            if file.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:\n                continue\n            target = root / label / file.relative_to(folder)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if not target.exists():\n                # Hard links avoid another large copy on cloud disk when supported.\n                try:\n                    os.link(file, target)\n                except OSError:\n                    shutil.copy2(file, target)\n            counts[label] += 1\n    (root / 'source.json').write_text(json.dumps({'source': DATASET_OPTIONS['plantifydr_pepper'],\n        'measured_file_counts': counts, 'parent_ids': 'Not inferred; review augmentation provenance'}, indent=2))\n    print('Measured Bell pepper files, before deduplication:', counts)\n    return root\n\nif __name__ == '__main__':\n    # Metadata only. Running this command never downloads images.\n    print(json.dumps(DATASET_OPTIONS, indent=2))\n",
  "fetch_mendeley.py": "\"\"\"Colab/cloud download of larger bell-pepper data, with optional six-class routing.\n\nThe upstream archive is about 705 MB. The claimed 9,283 count covers SIX classes;\nthe extracted healthy/bacterial-spot subset is smaller in binary mode. Its provenance/parent\ngroups still need review before treating any split as an independent benchmark.\n\"\"\"\nimport argparse\nimport http.client\nimport json\nimport shutil\nimport time\nimport urllib.error\nimport urllib.request\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nURL = 'https://data.mendeley.com/public-api/zip/w9drvtxg9g/download/1'\nFOLDERS = {'PepperBell_Bacterial Spot': 'bacterial_spot', 'PepperBell_Healthy': 'healthy'}\nALL_FOLDERS = dict(FOLDERS, **{'PepperBell_Cercospora Leaf Spot': 'cercospora_leaf_spot',\n    'PepperBell_Leaf Curl': 'leaf_curl', 'PepperBell_Nutrition Deficiency': 'nutrient_deficiency',\n    'PepperBell_Powdery Mildew': 'powdery_mildew'})\n\ndef download_zip(url, archive, attempts=3):\n    \"\"\"Retry bounded transient failures and resume only when the server honors Range.\"\"\"\n    archive = Path(archive)\n    partial = archive.with_suffix('.zip.part')\n    last_error = None\n    for attempt in range(attempts):\n        offset = partial.stat().st_size if partial.exists() else 0\n        headers = {'User-Agent': 'BellPepper-Coursework/1.0'}\n        if offset:\n            headers['Range'] = f'bytes={offset}-'\n        print(f'Dataset archive: attempt {attempt + 1}/{attempts}; cached partial {offset/1e6:.1f} MB', flush=True)\n        try:\n            request = urllib.request.Request(url, headers=headers)\n            with urllib.request.urlopen(request, timeout=90) as response:\n                append = bool(offset and response.status == 206)\n                if append and not response.headers.get('Content-Range', '').startswith(f'bytes {offset}-'):\n                    raise ValueError('The server returned an unexpected partial-file range.')\n                start = offset if append else 0\n                expected_chunk = int(response.headers.get('Content-Length', 0))\n                received = start\n                next_progress = start + 25 * 1024 * 1024\n                with partial.open('ab' if append else 'wb') as handle:\n                    while chunk := response.read(1024 * 1024):\n                        handle.write(chunk)\n                        received += len(chunk)\n                        if received >= next_progress:\n                            print(f'Downloaded {received/1e6:.0f} MB', flush=True)\n                            next_progress = received + 25 * 1024 * 1024\n                if expected_chunk and received - start != expected_chunk:\n                    raise OSError('Incomplete dataset transfer; retry will request the remaining bytes.')\n            if not zipfile.is_zipfile(partial):\n                raise ValueError('The server response was not a ZIP archive.')\n            partial.replace(archive)\n            return archive\n        except urllib.error.HTTPError as exc:\n            # A 416 can mean the partial file already contains the full archive.\n            if exc.code == 416 and partial.exists() and zipfile.is_zipfile(partial):\n                partial.replace(archive)\n                return archive\n            last_error = exc\n        except (OSError, ValueError, http.client.HTTPException) as exc:\n            last_error = exc\n        if attempt + 1 < attempts:\n            time.sleep(2 ** (attempt + 1))\n    raise RuntimeError(f'Automatic dataset download failed after {attempts} attempts: {last_error}. '\n                       f'Source API: {url}. Your existing files were preserved; rerun the data cell after connectivity recovers.') from last_error\n\ndef download(root='data/mendeley', archive_path='data/mendeley_v1.zip', task='multiclass', allow_local_download=False):\n    if task not in {'binary', 'multiclass'}:\n        raise ValueError('task must be binary or multiclass')\n    folders = ALL_FOLDERS if task == 'multiclass' else FOLDERS\n    root, archive = Path(root).resolve(), Path(archive_path)\n    archive.parent.mkdir(parents=True, exist_ok=True)\n    if archive.exists() and not zipfile.is_zipfile(archive):\n        archive.replace(archive.with_suffix('.invalid.zip'))\n    if not archive.exists():\n        from cloud_data import require_cloud_runtime\n        require_cloud_runtime(allow_local_download)\n        download_zip(URL, archive)\n    counts = dict.fromkeys(folders.values(), 0)\n    with zipfile.ZipFile(archive) as z:\n        for info in z.infolist():\n            parts = PurePosixPath(info.filename.replace('\\\\', '/')).parts\n            if info.is_dir() or Path(info.filename).suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:\n                continue\n            for folder, label in folders.items():\n                if folder not in parts:\n                    continue\n                relative = Path(*parts[parts.index(folder)+1:])\n                dest = (root / label / relative).resolve()\n                if not dest.is_relative_to(root / label):\n                    raise ValueError('Unsafe archive path.')\n                dest.parent.mkdir(parents=True, exist_ok=True)\n                if not dest.exists() or dest.stat().st_size != info.file_size:\n                    partial_file = dest.with_suffix(dest.suffix + '.part')\n                    with z.open(info) as source, partial_file.open('wb') as target:\n                        shutil.copyfileobj(source, target, length=1024*1024)\n                    partial_file.replace(dest)\n                counts[label] += 1\n    if not all(counts.values()):\n        raise RuntimeError(f'Expected class folders not found. Inspect archive layout. Counts: {counts}')\n    print('Extracted class file counts (before auditing duplicates):', counts)\n    (root / 'source.json').write_text(json.dumps({'source': 'https://data.mendeley.com/datasets/w9drvtxg9g/1',\n        'task': task, 'measured_file_counts': counts,\n        'parent_groups': 'Not inferred; review required for independent evaluation'}, indent=2), encoding='utf-8')\n    print('IMPORTANT: original/augmentation parent IDs are not supplied by this downloader. Audit before evaluation.')\n    return root\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--root', default='data/mendeley')\n    parser.add_argument('--archive', default='data/mendeley_v1.zip')\n    parser.add_argument('--task', choices=['binary', 'multiclass'], default='multiclass')\n    args = parser.parse_args()\n    download(args.root, args.archive, args.task)\n",
  "fetch_plantvillage.py": "\"\"\"Download only original colour bell-pepper images, with upstream leaf metadata.\"\"\"\nimport argparse\nimport concurrent.futures\nimport json\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nREPO = 'spMohanty/PlantVillage-Dataset'\nCLASSES = {'Pepper,_bell___Bacterial_spot': 'bacterial_spot',\n           'Pepper,_bell___healthy': 'healthy'}\n\ndef get(url):\n    for attempt in range(4):\n        try:\n            req = urllib.request.Request(url, headers={'User-Agent': 'BellPepper-Coursework/1.0'})\n            with urllib.request.urlopen(req, timeout=90) as response:\n                return response.read()\n        except Exception:\n            if attempt == 3:\n                raise\n            time.sleep(2 ** attempt)\n\ndef download(root='data/plantvillage', limit=None):\n    from cloud_data import require_cloud_runtime\n    require_cloud_runtime()\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    # Pin the resolved commit in provenance so a run can be reproduced.\n    sha = json.loads(get(f'https://api.github.com/repos/{REPO}/commits/master'))['sha']\n    base = f'https://raw.githubusercontent.com/{REPO}/{sha}/'\n    print(f'Using upstream commit {sha}', flush=True)\n    leaf_map = get(base + 'leaf-map.json')\n    (root / 'leaf-map.json').write_bytes(leaf_map)\n    folders = json.loads(get(f'https://api.github.com/repos/{REPO}/contents/raw/color?ref={sha}'))\n    selected = []\n    for folder, label in CLASSES.items():\n        folder_sha = next(e['sha'] for e in folders if e['name'] == folder)\n        tree = json.loads(get(f'https://api.github.com/repos/{REPO}/git/trees/{folder_sha}?recursive=1'))\n        if tree.get('truncated'):\n            raise RuntimeError('GitHub tree truncated; cannot claim a complete download.')\n        entries = [dict(e, path=f'raw/color/{folder}/' + e['path']) for e in tree['tree']\n                   if e['type'] == 'blob' and Path(e['path']).suffix.lower() in {'.jpg', '.jpeg', '.png'}]\n        selected.extend((e, label) for e in sorted(entries, key=lambda e: e['path'])[:limit])\n    if not selected:\n        raise RuntimeError('No matching pepper images in upstream repository.')\n    def one(item):\n        e, label = item\n        dest = root / label / Path(e['path']).name\n        dest.parent.mkdir(exist_ok=True)\n        if not dest.exists() or dest.stat().st_size != e['size']:\n            from urllib.parse import quote\n            data = get(base + quote(e['path'], safe='/'))\n            if len(data) != e['size']:\n                raise RuntimeError(f'Incomplete download: {dest.name}')\n            tmp = dest.with_suffix(dest.suffix + '.part')\n            tmp.write_bytes(data)\n            tmp.replace(dest)\n        return {'path': str(dest.resolve()), 'label': label, 'upstream_path': e['path']}\n    rows = []\n    with concurrent.futures.ThreadPoolExecutor(max_workers=6) as pool:\n        for row in pool.map(one, selected):\n            rows.append(row)\n            if len(rows) % 250 == 0:\n                print(f'Downloaded/verified {len(rows)}/{len(selected)}', flush=True)\n    (root / 'provenance.json').write_text(json.dumps({\n        'repository': REPO, 'commit': sha, 'downloaded_images': len(rows),\n        'partial_download': limit is not None, 'files': rows}, indent=2), encoding='utf-8')\n    print(f'Ready: {len(rows)} original colour images in {root}', flush=True)\n    return root\n\nif __name__ == '__main__':\n    p = argparse.ArgumentParser()\n    p.add_argument('--root', default='data/plantvillage')\n    p.add_argument('--limit', type=int, default=None, help='Smoke-test limit per class; omit for full data')\n    args = p.parse_args()\n    download(args.root, args.limit)\n",
  "pepper_model.py": "\"\"\"Shared RGB/HSV features, grouped evaluation, and seven classical classifiers.\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\n\nimport cv2\nimport joblib\nimport mahotas\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom PIL import Image, ImageOps\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\nfrom sklearn.tree import DecisionTreeClassifier\n\nVERSION = 'rgb-hsv-masked-532-v1'\nSIZE = 256\nLABELS = ['bacterial_spot', 'healthy']\nMULTICLASS_LABELS = ['bacterial_spot', 'cercospora_leaf_spot', 'healthy', 'leaf_curl',\n                     'nutrient_deficiency', 'powdery_mildew']\nMODEL_NAMES = ['Logistic Regression', 'Linear Discriminant Analysis', 'K-Nearest Neighbors',\n               'Decision Tree', 'Random Forest', 'Gaussian Naive Bayes', 'Support Vector Machine']\nDISPLAY_LABELS = {'bacterial_spot': 'Bacterial spot', 'healthy': 'Healthy',\n                  'cercospora_leaf_spot': 'Cercospora leaf spot', 'leaf_curl': 'Leaf curl',\n                  'nutrient_deficiency': 'Nutrient deficiency', 'powdery_mildew': 'Powdery mildew'}\n\ndef display_label(label):\n    return DISPLAY_LABELS.get(str(label), str(label).replace('_', ' ').title())\n\ndef condition_type(label):\n    if str(label).lower() in {'healthy', 'healthy_leaf'}:\n        return 'Healthy'\n    if str(label).lower() in {'nutrient_deficiency', 'nutrition_deficiency', 'magnesium_deficiency'}:\n        return 'Abiotic condition'\n    return 'Disease / symptom class'\n\ndef read_rgb(path):\n    with Image.open(path) as im:\n        return np.asarray(ImageOps.exif_transpose(im).convert('RGB'))\n\ndef preprocess(rgb):\n    rgb = np.asarray(rgb)\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:\n        raise ValueError('Expected an RGB uint8 image with three channels.')\n    rgb = cv2.resize(rgb, (SIZE, SIZE), interpolation=cv2.INTER_AREA)\n    smooth = cv2.GaussianBlur(rgb, (3, 3), 0.7)\n    hsv = cv2.cvtColor(smooth, cv2.COLOR_RGB2HSV)\n    # Colour-based candidate leaf region. This is not a lesion annotation.\n    green = cv2.inRange(hsv, (25, 35, 20), (100, 255, 255))\n    brown = cv2.inRange(hsv, (5, 35, 15), (30, 255, 255))\n    union = cv2.bitwise_or(green, brown)\n    union = cv2.morphologyEx(union, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))\n    contours, _ = cv2.findContours(union, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    mask = np.zeros(union.shape, np.uint8)\n    if contours:\n        # Filling the contour preserves dark / pale lesions inside the leaf.\n        cv2.drawContours(mask, [max(contours, key=cv2.contourArea)], -1, 255, cv2.FILLED)\n    coverage = float(np.count_nonzero(mask) / mask.size)\n    if coverage < 0.01:\n        raise ValueError('Too little usable leaf colour. Try a clear, close photo of one leaf.')\n    return rgb, smooth, hsv, mask, coverage\n\ndef extract_features(rgb):\n    rgb, smooth, hsv, mask, coverage = preprocess(rgb)\n    histogram = cv2.calcHist([hsv], [0, 1, 2], mask, [8, 8, 8], [0, 180, 0, 256, 0, 256])\n    histogram = histogram.ravel() / max(float(histogram.sum()), 1.0)\n    gray = cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY)\n    # Reserve zero for background; use 32 gray levels to keep GLCMs cheap.\n    quantized = np.where(mask > 0, gray // 8 + 1, 0).astype(np.uint8)\n    texture = mahotas.features.haralick(quantized, ignore_zeros=True, return_mean=True)\n    moments = cv2.HuMoments(cv2.moments(mask, binaryImage=True)).ravel()\n    moments = -np.sign(moments) * np.log10(np.maximum(np.abs(moments), 1e-30))\n    feature = np.concatenate([histogram, texture, moments]).astype(np.float32)\n    if feature.shape != (532,) or not np.isfinite(feature).all():\n        raise ValueError('Invalid feature vector; inspect the input image.')\n    return feature\n\ndef make_manifest(root, output, groups_csv=None, class_names=None, conflict_policy='exclude'):\n    \"\"\"Index files without copying, renaming, or deleting any source image.\n\n    groups_csv: path,group_id columns; paths relative to root or absolute.\n    Group IDs must identify the same plant/source image across derivatives.\n    Conflicting decoded duplicates are reported and all copies are excluded by\n    default. 'error' writes the same audit, then stops for manual review.\n    \"\"\"\n    root, output = Path(root).resolve(), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if conflict_policy not in {'exclude', 'error'}:\n        raise ValueError(\"conflict_policy must be 'exclude' or 'error'.\")\n    manual = {}\n    if groups_csv:\n        for r in pd.read_csv(groups_csv, dtype=str).itertuples():\n            manual[str((root / r.path).resolve())] = r.group_id\n    leaf_map_file = root / 'leaf-map.json'\n    leaf_map = json.loads(leaf_map_file.read_text()) if leaf_map_file.exists() else {}\n    upstream_classes = {'bacterial_spot': 'Pepper,_bell___Bacterial_spot',\n                        'healthy': 'Pepper,_bell___healthy'}\n    class_names = list(class_names) if class_names is not None else LABELS\n    if len(class_names) < 2 or len(set(class_names)) != len(class_names):\n        raise ValueError('Provide at least two distinct class names.')\n    rows, rejected = [], []\n    for label in class_names:\n        folder = root / label\n        if not folder.is_dir():\n            raise ValueError(f'Missing class folder: {folder}')\n        for path in sorted(folder.rglob('*')):\n            if path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:\n                continue\n            try:\n                rgb = read_rgb(path)\n                digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()\n                tiny = cv2.resize(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (32, 32))\n                dct = cv2.dct(tiny.astype(np.float32))[:8, :8].ravel()[1:]\n                phash = ''.join('1' if v > np.median(dct) else '0' for v in dct)\n                identifier = path.stem.replace('_final_masked', '').split('___')[-1].split('copy')[0].strip().lower()\n                suggestions = leaf_map.get(identifier, [])\n                upstream_class = upstream_classes.get(label)\n                matches = [s for s in suggestions if upstream_class and upstream_class in s]\n                leaf_id = suggestions[0] if len(suggestions) == 1 else (matches[0] if len(matches) == 1 else None)\n                source_group = manual.get(str(path), leaf_id or digest)\n                basis = 'manual' if str(path) in manual else ('upstream_leaf_id' if leaf_id else 'image_hash_only')\n                rows.append({'path': str(path), 'label': label, 'sha256': digest,\n                             'phash': phash, 'group_id': source_group,\n                             'source_group': source_group, 'group_basis': basis})\n            except Exception as exc:\n                rejected.append({'path': str(path), 'reason': str(exc)})\n    df = pd.DataFrame(rows)\n    if df.empty:\n        raise ValueError('No readable images found.')\n    missing = set(class_names) - set(df.label)\n    if missing:\n        raise ValueError(f'No readable images in requested class(es): {sorted(missing)}')\n    label_counts = df.groupby('sha256').label.nunique()\n    conflict_hashes = set(label_counts[label_counts > 1].index)\n    conflicts = df[df.sha256.isin(conflict_hashes)].copy()\n    label_sets = df.groupby('sha256').label.agg(lambda values: ' | '.join(sorted(set(values))))\n    conflicts['conflicting_labels'] = conflicts.sha256.map(label_sets)\n    conflicts['action'] = 'excluded_from_training_and_evaluation' if conflict_policy == 'exclude' else 'requires_manual_review'\n    conflicts.to_csv(output / 'label_conflicts.csv', index=False)\n    pd.DataFrame(rejected, columns=['path', 'reason']).to_csv(output / 'unreadable.csv', index=False)\n    df[df.duplicated('sha256', keep=False)].to_csv(output / 'exact_duplicates.csv', index=False)\n    counts_before = {label: int(df.label.eq(label).sum()) for label in class_names}\n    audit = {\n        'conflict_policy': conflict_policy,\n        'readable_files_before_cleanup': len(df), 'unreadable_files': len(rejected),\n        'conflicting_image_groups': len(conflict_hashes), 'conflicting_files_detected': len(conflicts),\n        'conflicting_files_excluded': len(conflicts) if conflict_policy == 'exclude' else 0,\n        'counts_before_cleanup': counts_before,\n    }\n    if conflict_hashes and conflict_policy == 'error':\n        audit['status'] = 'blocked_for_label_review'\n        (output / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')\n        raise ValueError(f'Identical decoded images have conflicting labels: {len(conflicts)} files '\n                         f'in {len(conflict_hashes)} groups. See {output / \"label_conflicts.csv\"}. '\n                         \"Use conflict_policy='exclude' to omit every copy without changing source files.\")\n    # Connect matching metadata groups, exact images, and exact perceptual hashes.\n    # This is a conservative heuristic, NOT a complete near-duplicate detector.\n    parent = list(range(len(df)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n    for column in ['group_id', 'sha256', 'phash']:\n        first = {}\n        for i, key in enumerate(df[column]):\n            if key in first:\n                parent[find(i)] = find(first[key])\n            else:\n                first[key] = i\n    df['group_id'] = [f'group-{find(i)}' for i in range(len(df))]\n    # Form groups BEFORE exclusion/deduplication so removed rows cannot break a\n    # connection between surviving parent groups or transformed siblings.\n    unambiguous = df[~df.sha256.isin(conflict_hashes)].copy()\n    df = unambiguous.drop_duplicates('sha256').reset_index(drop=True)\n    audit['same_label_duplicate_copies_removed'] = len(unambiguous) - len(df)\n    audit['retained_unique_images'] = len(df)\n    audit['counts_after_cleanup'] = {label: int(df.label.eq(label).sum()) for label in class_names}\n    missing_after_cleanup = set(class_names) - set(df.label)\n    audit['status'] = 'blocked_missing_classes_after_cleanup' if missing_after_cleanup else 'ready'\n    (output / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')\n    df.to_csv(output / 'manifest.csv', index=False)\n    if missing_after_cleanup:\n        raise ValueError('No usable images remain after label-conflict exclusion in class(es): '\n                         f'{sorted(missing_after_cleanup)}. Review {output / \"label_conflicts.csv\"}; '\n                         'add correctly labelled examples before training.')\n    if conflict_hashes:\n        print(f'[AUDIT] Excluded all {len(conflicts)} files in {len(conflict_hashes)} conflicting-label '\n              'groups. Source images were preserved. See label_conflicts.csv in the output folder.', flush=True)\n    print(f'[AUDIT] Retained {len(df)} unique unambiguous images. '\n          f'Per class: {audit[\"counts_after_cleanup\"]}', flush=True)\n    df.attrs['data_audit'] = audit\n    return df\n\ndef assign_splits(df, seed=42):\n    df = df.copy()\n    labels = set(df.label)\n    if df.groupby('label').group_id.nunique().min() < 5:\n        raise ValueError('Need at least five independent groups per class.')\n    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n    df['split'] = 'train'\n    for fold, (_, indices) in enumerate(folds.split(df, df.label, df.group_id)):\n        if fold in (0, 1):\n            df.loc[df.index[indices], 'split'] = 'test' if fold == 0 else 'validation'\n    if df.groupby('group_id').split.nunique().max() != 1:\n        raise AssertionError('A group crosses partitions.')\n    for split in ['train', 'validation', 'test']:\n        if set(df.loc[df.split == split, 'label']) != labels:\n            raise ValueError(f'{split} lacks one class. Review group sizes before changing the seed.')\n    return df\n\ndef cached_features(df, output):\n    output = Path(output)\n    cache = output / 'feature_cache' / VERSION\n    cache.mkdir(parents=True, exist_ok=True)\n    matrix = np.lib.format.open_memmap(output / 'features.npy', mode='w+', dtype='float32', shape=(len(df), 532))\n    failures, valid = [], np.ones(len(df), dtype=bool)\n    for i, r in enumerate(df.itertuples()):\n        file = cache / (r.sha256 + '.npy')\n        try:\n            feature = None\n            if file.exists():\n                try:\n                    candidate = np.load(file, allow_pickle=False)\n                    if candidate.shape == (532,) and np.isfinite(candidate).all():\n                        feature = candidate\n                except (OSError, ValueError, EOFError):\n                    pass  # interrupted/corrupt cache: recover from the unchanged source image\n            if feature is None:\n                feature = extract_features(read_rgb(r.path))\n                temp = file.with_suffix('.tmp')\n                with temp.open('wb') as handle:\n                    np.save(handle, feature)\n                temp.replace(file)\n            matrix[i] = feature\n        except Exception as exc:\n            valid[i] = False\n            matrix[i] = 0\n            failures.append({'path': r.path, 'label': r.label, 'split': r.split, 'reason': str(exc)})\n        if (i + 1) % 250 == 0:\n            print(f'Features {i+1}/{len(df)}', flush=True)\n    matrix.flush()\n    pd.DataFrame(failures, columns=['path', 'label', 'split', 'reason']).to_csv(output / 'feature_failures.csv', index=False)\n    return matrix, valid\n\ndef training_views(df, X, mask, output, target_rows=0):\n    \"\"\"Optional train-only synthetic views; never manufacture validation/test rows.\"\"\"\n    originals = df.loc[mask].reset_index(drop=True)\n    if target_rows <= len(originals):\n        return X[mask], originals.label.to_numpy()\n    rng = np.random.default_rng(42)\n    expanded = np.lib.format.open_memmap(Path(output) / 'training_views.npy', mode='w+',\n                                        dtype='float32', shape=(target_rows, 532))\n    expanded[:len(originals)] = X[mask]\n    labels = originals.label.tolist()\n    parents = originals.sha256.tolist()\n    attempts = 0\n    while len(labels) < target_rows:\n        attempts += 1\n        if attempts > target_rows * 3:\n            raise RuntimeError('Too many augmentation failures. Review segmentation.')\n        # Balance synthetic views, while retaining every original training image.\n        available_labels = sorted(originals.label.unique())\n        label = available_labels[len(labels) % len(available_labels)]\n        pool = originals[originals.label == label]\n        row = pool.iloc[int(rng.integers(len(pool)))]\n        rgb = np.rot90(read_rgb(row.path), int(rng.integers(4)))\n        if rng.random() < 0.5:\n            rgb = np.fliplr(rgb)\n        rgb = np.clip(rgb.astype(np.float32) * rng.uniform(0.9, 1.1), 0, 255).astype(np.uint8)\n        try:\n            feature = extract_features(np.ascontiguousarray(rgb))\n        except ValueError:\n            continue\n        expanded[len(labels)] = feature\n        labels.append(label)\n        parents.append(row.sha256)\n        if len(labels) % 1000 == 0:\n            expanded.flush()\n            print(f'Training views {len(labels)}/{target_rows}', flush=True)\n    expanded.flush()\n    pd.DataFrame({'label': labels, 'parent_sha256': parents}).to_csv(Path(output) / 'training_views.csv', index=False)\n    return expanded, np.asarray(labels)\n\ndef wilson(correct, total):\n    if not total:\n        return [None, None]\n    z = 1.96\n    p = correct / total\n    center = (p + z*z/(2*total)) / (1+z*z/total)\n    margin = z * np.sqrt(p*(1-p)/total + z*z/(4*total*total)) / (1+z*z/total)\n    return [float(center-margin), float(center+margin)]\n\ndef make_models(seed=42):\n    \"\"\"All seven original algorithm families; scaling is fitted on training only.\n\n    LDA uses shrinkage to handle correlated/constant histogram columns. SVM\n    deliberately has no internally fitted probability calibration; it votes using\n    predict(), so ungrouped calibration folds cannot leak leaf identities.\n    \"\"\"\n    return {\n        'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(\n            max_iter=3000, class_weight='balanced', random_state=seed)),\n        'Linear Discriminant Analysis': make_pipeline(StandardScaler(),\n            LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),\n        'K-Nearest Neighbors': make_pipeline(StandardScaler(),\n            KNeighborsClassifier(n_neighbors=7, weights='distance', algorithm='ball_tree', n_jobs=2)),\n        'Decision Tree': DecisionTreeClassifier(min_samples_leaf=2,\n            class_weight='balanced', random_state=seed),\n        'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12,\n            min_samples_leaf=2, class_weight='balanced', n_jobs=2, random_state=seed),\n        'Gaussian Naive Bayes': make_pipeline(StandardScaler(), GaussianNB(var_smoothing=1e-8)),\n        'Support Vector Machine': make_pipeline(StandardScaler(),\n            SVC(C=5, class_weight='balanced', random_state=seed)),\n    }\n\ndef combine_votes(predictions, classes):\n    \"\"\"Require a strict majority. Multi-class plurality/ties are inconclusive.\"\"\"\n    if not predictions:\n        raise ValueError('No classifier predictions available.')\n    classes = list(classes)\n    if len(set(classes)) != len(classes) or len(classes) < 2:\n        raise ValueError('Invalid class list.')\n    if any(p not in classes for p in predictions):\n        raise ValueError('A classifier returned a class outside the bundle taxonomy.')\n    counts = {label: predictions.count(label) for label in classes}\n    maximum = max(counts.values())\n    leaders = [label for label in classes if counts[label] == maximum]\n    majority = len(leaders) == 1 and maximum > len(predictions) / 2\n    return {'prediction': leaders[0] if majority else None, 'votes': counts,\n            'votes_for_leading_class': maximum, 'total_models': len(predictions),\n            'agreement': maximum / len(predictions), 'leading_classes': leaders,\n            'status': 'majority' if majority else 'inconclusive',\n            'method': 'equal hard voting, strict majority required; agreement is not a probability'}\n\ndef predict_all(bundle, feature):\n    \"\"\"One feature extraction, all fitted classifiers, class-aware overall vote.\"\"\"\n    if bundle.get('feature_version') != VERSION:\n        raise ValueError('Feature version mismatch. Retrain with this feature code.')\n    models = bundle.get('models')\n    if models is None and 'model' in bundle:\n        models = {'Random Forest': bundle['model']}\n    if not models:\n        raise ValueError('No trained classifiers found in the bundle.')\n    feature = np.asarray(feature, dtype=np.float32).reshape(1, -1)\n    if feature.shape != (1, 532) or not np.isfinite(feature).all():\n        raise ValueError('Expected a finite 532-feature vector.')\n    classes = [str(c) for c in bundle['classes']]\n    rows, predictions = [], []\n    for name, model in models.items():\n        if set(map(str, model.classes_)) != set(classes):\n            raise ValueError(f'{name} was trained on different classes.')\n        predicted = str(model.predict(feature)[0])\n        score = None\n        if hasattr(model, 'predict_proba'):\n            probabilities = model.predict_proba(feature)[0]\n            score = float(probabilities[list(map(str, model.classes_)).index(predicted)])\n        rows.append({'model': name, 'prediction': predicted,\n                     'condition': display_label(predicted), 'condition_type': condition_type(predicted),\n                     'model_score': score,\n                     'score_note': 'uncalibrated class score' if score is not None else 'SVM: class vote only'})\n        predictions.append(predicted)\n    vote_comparison = combine_votes(predictions, classes)\n    overall = dict(vote_comparison)\n    selected_name = bundle.get('selected_model')\n    if selected_name is not None:\n        if selected_name not in models:\n            raise ValueError('The validation-selected classifier is missing from the bundle.')\n        selected_row = next(r for r in rows if r['model'] == selected_name)\n        overall.update(prediction=selected_row['prediction'], status='selected_model',\n                       selected_model=selected_name, model_score=selected_row['model_score'],\n                       selection_metric=bundle.get('selection_metric'),\n                       selection_score=bundle.get('selection_score'),\n                       method='validation_selected_model',\n                       votes_for_selected_class=vote_comparison['votes'][selected_row['prediction']])\n    overall['seven_model_comparison'] = set(models) == set(MODEL_NAMES)\n    if overall['prediction']:\n        overall['condition'] = display_label(overall['prediction'])\n        overall['condition_type'] = condition_type(overall['prediction'])\n    else:\n        overall['condition'] = 'Inconclusive \u2014 classifiers disagree'\n        overall['condition_type'] = 'Undetermined'\n    return {'per_model': rows, 'overall': overall, 'vote_comparison': vote_comparison}\n\ndef evaluate_predictions(actual, predicted, classes):\n    return {\n        'accuracy': float(accuracy_score(actual, predicted)),\n        'balanced_accuracy': float(balanced_accuracy_score(actual, predicted)),\n        'classification_report': classification_report(actual, predicted, labels=classes,\n            output_dict=True, zero_division=0),\n        'confusion_matrix': confusion_matrix(actual, predicted, labels=classes).tolist(),\n    }\n\ndef train(root, output='artifacts_all_models', groups_csv=None, target_train_views=0,\n          class_names=None, feature_output=None, data_note=None, conflict_policy='exclude'):\n    \"\"\"Fit seven models and select the primary classifier on validation only.\n\n    feature_output optionally reuses an existing feature cache without touching\n    the previous model bundle. No classifier is selected using the test results.\n    \"\"\"\n    output = Path(output)\n    df = assign_splits(make_manifest(root, output, groups_csv, class_names, conflict_policy))\n    classes = sorted(df.label.unique().tolist())\n    df.to_csv(output / 'manifest.csv', index=False)\n    if feature_output:\n        feature_root = Path(feature_output)\n        feature_root.mkdir(parents=True, exist_ok=True)\n    else:\n        feature_root = output\n    X, valid = cached_features(df, feature_root)\n    if feature_root != output:\n        (output / 'feature_failures.csv').write_bytes((feature_root / 'feature_failures.csv').read_bytes())\n    masks = {s: (df.split.eq(s).to_numpy() & valid) for s in ['train', 'validation', 'test']}\n    for split, mask in masks.items():\n        if set(df.loc[mask, 'label']) != set(classes):\n            raise ValueError(f'{split} lacks a class after feature failures.')\n    y = df.label.to_numpy()\n    train_X, train_y = training_views(df, X, masks['train'], output, target_train_views)\n    models = make_models()\n    validation, test_predictions = {}, {}\n    for name, model in models.items():\n        print(f'Training {name} on {len(train_y)} rows...', flush=True)\n        model.fit(train_X, train_y)\n        val_pred = model.predict(X[masks['validation']])\n        validation[name] = evaluate_predictions(y[masks['validation']], val_pred, classes)\n    selected_name = max(models, key=lambda name: validation[name]['balanced_accuracy'])\n    selection_score = validation[selected_name]['balanced_accuracy']\n    # Freeze the choice before any test predictions. Ties follow declared model order.\n    print(f'Selected on validation balanced accuracy: {selected_name} ({selection_score:.4f})', flush=True)\n    test = masks['test']\n    per_model_metrics = {}\n    for name, model in models.items():\n        prediction = model.predict(X[test])\n        test_predictions[name] = prediction\n        per_model_metrics[name] = evaluate_predictions(y[test], prediction, classes)\n    votes = [combine_votes([str(test_predictions[name][i]) for name in models], classes)\n             for i in range(int(test.sum()))]\n    vote_predictions = np.array([r['prediction'] or '__inconclusive__' for r in votes])\n    overall_predictions = np.asarray(test_predictions[selected_name])\n    decided = vote_predictions != '__inconclusive__'\n    correct = int((overall_predictions == y[test]).sum())\n    n_total = int(df.split.eq('test').sum())\n    report = {\n        'feature_version': VERSION, 'bundle_version': 3, 'labels': classes,\n        'selected_model': selected_name, 'selection_metric': 'validation_balanced_accuracy',\n        'selection_score': float(selection_score),\n        'primary_prediction_rule': 'Use the individual model selected on validation; no test-based selection',\n        'model_names': list(models), 'ensemble_rule': 'Equal hard vote; strict majority required',\n        'split_counts': pd.crosstab(df.split, df.label).to_dict(),\n        'unique_images': len(df), 'feature_failures': int((~valid).sum()),\n        'data_audit': df.attrs['data_audit'],\n        'training_rows_including_synthetic_views': len(train_y),\n        'grouping_basis_counts': df.group_basis.value_counts().to_dict(),\n        'validation_per_model': validation, 'test_per_model': per_model_metrics,\n        'test_total': n_total, 'test_accepted': int(test.sum()),\n        'test_coverage': float(test.sum()/n_total),\n        'ensemble_decided': int(decided.sum()),\n        'ensemble_decision_coverage_on_accepted': float(decided.mean()),\n        'test_accuracy_on_accepted': correct / int(test.sum()),\n        'test_accuracy_counting_rejections_as_errors': correct / n_total,\n        'ensemble_accuracy_on_decided': float(accuracy_score(y[test][decided], vote_predictions[decided])) if decided.any() else None,\n        'ensemble_accuracy_counting_rejections_and_abstentions_as_errors': int((vote_predictions == y[test]).sum()) / n_total,\n        'test_balanced_accuracy': per_model_metrics[selected_name]['balanced_accuracy'],\n        'image_level_wilson_95': wilson(correct, int(test.sum())),\n        'ci_note': 'Image-level interval; correlated photos reduce effective sample size.',\n        'classification_report': classification_report(y[test], overall_predictions,\n            labels=classes, output_dict=True, zero_division=0),\n        'confusion_matrix': confusion_matrix(y[test], overall_predictions,\n            labels=classes + ['__inconclusive__']).tolist(),\n        'data_note': data_note or 'Dataset provenance and parent grouping require review.',\n        'limitations': 'Internal comparison only. Hashes do not prove independence. Disease names are restricted to trained labels. Field accuracy is unmeasured.',\n    }\n    versions = {'python': platform.python_version(), 'numpy': np.__version__,\n                'opencv': cv2.__version__, 'sklearn': sklearn.__version__, 'mahotas': mahotas.__version__}\n    bundle = {'models': models, 'bundle_version': 3, 'feature_version': VERSION,\n              'selected_model': selected_name, 'selection_metric': report['selection_metric'],\n              'selection_score': float(selection_score),\n              'size': SIZE, 'classes': classes, 'versions': versions,\n              'ensemble_rule': report['ensemble_rule'], 'data_note': report['data_note']}\n    joblib.dump(bundle, output / 'pepper_models.joblib')\n    (output / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')\n    (output / 'environment.json').write_text(json.dumps(versions, indent=2), encoding='utf-8')\n    result = df.loc[test, ['path', 'label', 'group_id']].copy()\n    for name, prediction in test_predictions.items():\n        result[name] = prediction\n    result['overall_prediction'] = overall_predictions\n    result['vote_prediction'] = vote_predictions\n    result['vote_agreement'] = [r['agreement'] for r in votes]\n    result.to_csv(output / 'test_predictions.csv', index=False)\n    table = pd.DataFrame([{'model': name, 'validation_balanced_accuracy': validation[name]['balanced_accuracy'],\n                           'validation_macro_f1': validation[name]['classification_report']['macro avg']['f1-score'],\n                           'test_accuracy': metrics['accuracy'], 'test_balanced_accuracy': metrics['balanced_accuracy'],\n                           'test_macro_f1': metrics['classification_report']['macro avg']['f1-score'],\n                           'selected_for_main_prediction': name == selected_name}\n                          for name, metrics in per_model_metrics.items()])\n    table.to_csv(output / 'model_comparison.csv', index=False)\n    print(table.to_string(index=False), flush=True)\n    print(f'Primary selected model ({selected_name}): {correct}/{n_total} correct. '\n          f'Optional voting decisions: {int(decided.sum())}/{int(test.sum())}', flush=True)\n    return report\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--data', default='data/plantvillage')\n    parser.add_argument('--output', default='artifacts_all_models')\n    parser.add_argument('--groups', default=None)\n    parser.add_argument('--train-views', type=int, default=0, help='Optional total training views; synthetic, not independent images')\n    parser.add_argument('--classes', nargs='+', default=None, help='Class folder names; defaults to bacterial_spot healthy')\n    parser.add_argument('--feature-output', default=None, help='Existing cache to reuse without altering the previous trained bundle')\n    parser.add_argument('--data-note', default=None)\n    parser.add_argument('--conflict-policy', choices=['exclude', 'error'], default='exclude')\n    args = parser.parse_args()\n    train(args.data, args.output, args.groups, args.train_views, args.classes, args.feature_output, args.data_note, args.conflict_policy)\n",
  "notebook_ui.py": "\"\"\"Notebook-native prediction panel. No web server or public sharing required.\"\"\"\nimport html\nimport io\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageOps\nfrom pepper_model import extract_features, preprocess, predict_all\n\ndef prediction_for_bytes(content, bundle):\n    with Image.open(io.BytesIO(bytes(content))) as image:\n        rgb = np.asarray(ImageOps.exif_transpose(image).convert('RGB'))\n    prediction = predict_all(bundle, extract_features(rgb))\n    rows = []\n    for row in prediction['per_model']:\n        rows.append({'Classifier': row['model'], 'Prediction': row['condition'],\n                     'Category': row['condition_type'],\n                     'Model score (uncalibrated)': f\"{row['model_score']:.1%}\" if row['model_score'] is not None else 'Not supplied',\n                     'Main classifier': 'Yes' if row['model'] == prediction['overall'].get('selected_model') else ''})\n    rgb, _, hsv, mask, coverage = preprocess(rgb)\n    segmented = rgb.copy()\n    segmented[mask == 0] = 0\n    stages = [Image.fromarray(rgb), Image.fromarray(hsv[:, :, 0]).convert('RGB'),\n              Image.fromarray(mask).convert('RGB'), Image.fromarray(segmented)]\n    labels = ['RGB input', 'Hue channel', 'Candidate leaf mask', 'Selected pixels']\n    sheet = Image.new('RGB', (512, 568), '#f3f6ed')\n    draw = ImageDraw.Draw(sheet)\n    for i, (stage, label) in enumerate(zip(stages, labels)):\n        x, y = (i % 2) * 256, (i // 2) * 284\n        draw.text((x + 9, y + 8), label, fill='#173c2c')\n        sheet.paste(stage, (x, y + 28))\n    prediction['mask_coverage'] = coverage\n    return pd.DataFrame(rows), prediction, sheet\n\ndef build_notebook_ui(bundle, output_dir=None):\n    import ipywidgets as widgets\n    from IPython.display import HTML, display\n\n    title = widgets.HTML('<div style=\"padding:22px;background:#173c2c;color:white;border-radius:14px\">'\n                         '<h2 style=\"margin:0;color:white\">PepperLens</h2>'\n                         '<p>Upload one bell pepper leaf. Compare seven classifiers and see the validation-selected result.</p></div>')\n    upload = widgets.FileUpload(accept='image/*', multiple=False, description='Choose leaf image')\n    analyze = widgets.Button(description='Analyze leaf', button_style='success', disabled=True,\n                             layout=widgets.Layout(width='180px'))\n    output = widgets.Output()\n    note = widgets.HTML('Upload only your new prediction photograph. The training dataset is fetched automatically. '\n                        'Scores are uncalibrated; crop identity and unseen diseases are not verified.')\n\n    def uploaded_file():\n        if isinstance(upload.value, dict):  # ipywidgets 7 compatibility\n            name, entry = next(iter(upload.value.items()))\n            return name, entry['content']\n        entry = upload.value[0]  # ipywidgets 8\n        return entry.get('name', 'leaf'), entry['content']\n\n    def analyze_clicked(_):\n        with output:\n            output.clear_output(wait=True)\n            if not upload.value:\n                print('Choose a leaf photograph first.')\n                return\n            try:\n                name, content = uploaded_file()\n                table, prediction, stages = prediction_for_bytes(content, bundle)\n                main = prediction['overall']\n                model_name = main.get('selected_model', 'legacy bundle')\n                display(HTML('<div style=\"padding:16px;background:#e5efdf;border-radius:12px;color:#173c2c\">'\n                             f'<h3 style=\"margin:0\">Main prediction: {html.escape(main[\"condition\"])}</h3>'\n                             f'<p>Classifier: <b>{html.escape(model_name)}</b> \u00b7 {html.escape(main[\"condition_type\"])}</p>'\n                             '</div>'))\n                display(table)\n                display(stages)\n                print(f'Candidate mask coverage: {prediction[\"mask_coverage\"]:.1%}. This is not disease severity.')\n                if output_dir:\n                    import json\n                    destination = Path(output_dir)\n                    destination.mkdir(parents=True, exist_ok=True)\n                    (destination / 'latest_prediction.json').write_text(\n                        json.dumps({'image_name': str(name), **prediction}, indent=2), encoding='utf-8')\n                    print('Saved latest_prediction.json in the results folder.')\n            except (ValueError, OSError) as exc:\n                print(f'Unable to analyze that image: {exc}')\n\n    upload.observe(lambda change: setattr(analyze, 'disabled', not bool(change['new'])), names='value')\n    analyze.on_click(analyze_clicked)\n    return widgets.VBox([title, note, widgets.HBox([upload, analyze]), output])\n",
  "dsp_walkthrough.py": "\"\"\"Coursework plots of the shared feature pipeline; FFT/Sobel are diagnostics only.\"\"\"\nimport cv2\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom pepper_model import preprocess, extract_features\n\n\ndef plot_processing_stages(image):\n    \"\"\"Show channel order, colour coordinates, nonlinear masks and segmentation.\"\"\"\n    rgb, smooth, hsv, mask, coverage = preprocess(image)\n    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)\n    restored = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n    np.testing.assert_array_equal(restored, rgb)\n    green = cv2.inRange(hsv, (25, 35, 20), (100, 255, 255))\n    brown = cv2.inRange(hsv, (5, 35, 15), (30, 255, 255))\n    union = cv2.bitwise_or(green, brown)\n    closed = cv2.morphologyEx(union, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))\n    segmented = rgb.copy()\n    segmented[mask == 0] = 0\n    gray = cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY)\n    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)\n    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)\n    edge_magnitude = np.hypot(gx, gy)\n    log_spectrum = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray))))\n    stages = [\n        (rgb, '1. Resampled RGB: 256 x 256', None, None),\n        (bgr, '2. BGR incorrectly shown as RGB\\n(channel-order demonstration)', None, None),\n        (restored, '3. Correct BGR -> RGB display', None, None),\n        (smooth, '4. Gaussian smoothing\\n3 x 3, sigma = 0.7 pixels', None, None),\n        (hsv[:, :, 0], '5. Hue H: OpenCV range 0-179', 'hsv', (0, 179)),\n        (hsv[:, :, 1], '6. Saturation S: 0-255', 'gray', (0, 255)),\n        (hsv[:, :, 2], '7. Value V: 0-255', 'gray', (0, 255)),\n        (gray, '8. Grayscale for texture', 'gray', (0, 255)),\n        (green, '9. Green threshold mask', 'gray', (0, 255)),\n        (brown, '10. Brown threshold mask', 'gray', (0, 255)),\n        (union, '11. Mask union: bitwise OR', 'gray', (0, 255)),\n        (closed, '12. Morphological closing', 'gray', (0, 255)),\n        (mask, '13. Filled largest contour', 'gray', (0, 255)),\n        (segmented, '14. Selected leaf pixels', None, None),\n        (edge_magnitude, '15. Sobel edge magnitude\\n(diagnostic only)', 'magma', None),\n        (log_spectrum, '16. Log magnitude of 2-D DFT\\n(diagnostic only)', 'magma', None),\n    ]\n    fig, axes = plt.subplots(4, 4, figsize=(15, 15))\n    for axis, (pixels, title, colour_map, limits) in zip(axes.flat, stages):\n        options = {'cmap': colour_map} if colour_map else {}\n        if limits:\n            options.update(vmin=limits[0], vmax=limits[1])\n        axis.imshow(pixels, **options)\n        axis.set_title(title, fontsize=10)\n        axis.axis('off')\n    fig.suptitle(f'Actual leaf-processing pipeline; candidate mask coverage {coverage:.1%}', fontsize=15)\n    fig.tight_layout(rect=(0, 0, 1, .97))\n    arrays = {'rgb': rgb, 'bgr': bgr, 'smooth': smooth, 'hsv': hsv, 'mask': mask,\n              'green': green, 'brown': brown, 'closed': closed, 'segmented': segmented,\n              'edge_magnitude': edge_magnitude, 'log_spectrum': log_spectrum}\n    return fig, arrays\n\n\ndef plot_filter_analysis(image):\n    \"\"\"Relate the Gaussian convolution used by the model to its frequency response.\"\"\"\n    rgb, smooth, _, _, _ = preprocess(image)\n    gray_before = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)\n    gray_after = cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY).astype(np.float64)\n    gaussian_1d = cv2.getGaussianKernel(3, .7)\n    kernel = gaussian_1d @ gaussian_1d.T\n    response = np.abs(np.fft.fftshift(np.fft.fft2(kernel, s=gray_before.shape)))\n    frequency = np.fft.fftshift(np.fft.fftfreq(gray_before.shape[1]))\n    before_spectrum = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray_before))))\n    after_spectrum = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray_after))))\n    fig, axes = plt.subplots(2, 3, figsize=(14, 8))\n    axes[0, 0].imshow(kernel, cmap='Greens')\n    for i in range(3):\n        for j in range(3):\n            axes[0, 0].text(j, i, f'{kernel[i, j]:.3f}', ha='center', va='center')\n    axes[0, 0].set_title('Normalized Gaussian convolution kernel')\n    axes[0, 1].imshow(response, cmap='viridis', extent=(-.5, .5, -.5, .5), vmin=0, vmax=1)\n    axes[0, 1].set_title('Magnitude frequency response |H(fx, fy)|')\n    axes[0, 1].set_xlabel('Horizontal cycles / pixel')\n    axes[0, 1].set_ylabel('Vertical cycles / pixel')\n    axes[0, 2].plot(frequency, response[response.shape[0]//2])\n    axes[0, 2].set_title('Filter response along fy = 0')\n    axes[0, 2].set_xlabel('Horizontal cycles / pixel')\n    axes[0, 2].set_ylabel('Amplitude gain')\n    axes[0, 2].set_ylim(0, 1.05)\n    axes[0, 2].grid(alpha=.3)\n    shared_max = max(before_spectrum.max(), after_spectrum.max())\n    axes[1, 0].imshow(before_spectrum, cmap='magma', vmin=0, vmax=shared_max)\n    axes[1, 0].set_title('Log DFT magnitude before smoothing')\n    axes[1, 1].imshow(after_spectrum, cmap='magma', vmin=0, vmax=shared_max)\n    axes[1, 1].set_title('Log DFT magnitude after smoothing\\n(same display scale)')\n    residual = gray_before - gray_after\n    limit = max(float(np.abs(residual).max()), 1)\n    axes[1, 2].imshow(residual, cmap='coolwarm', vmin=-limit, vmax=limit)\n    axes[1, 2].set_title('Removed spatial detail: before - after')\n    for axis in [axes[0, 0], *axes[1]]:\n        axis.axis('off')\n    fig.suptitle('DSP application: spatial low-pass filtering and its 2-D frequency response')\n    fig.tight_layout(rect=(0, 0, 1, .95))\n    return fig, kernel, response\n\n\ndef plot_feature_summary(image):\n    \"\"\"Visualize the same 532 values supplied to every classifier.\"\"\"\n    feature = extract_features(image)\n    histogram = feature[:512].reshape(8, 8, 8)\n    texture = feature[512:525]\n    hu = feature[525:532]\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4))\n    for axis, marginal, title, upper in zip(\n            axes, [histogram.sum(axis=(1, 2)), histogram.sum(axis=(0, 2)), histogram.sum(axis=(0, 1))],\n            ['Hue marginal', 'Saturation marginal', 'Value marginal'], [180, 256, 256]):\n        centers = (np.arange(8) + .5) * upper / 8\n        axis.bar(centers, marginal, width=upper/10, color='#36734b')\n        axis.set_title(title + ' of masked 8 x 8 x 8 histogram')\n        axis.set_xlabel('Channel-bin center')\n        axis.set_ylabel('Fraction of selected pixels')\n        axis.set_xlim(0, upper)\n    fig.tight_layout()\n    texture_table = pd.DataFrame({'Descriptor': [f'Haralick {i+1}' for i in range(13)],\n                                  'Actual feature value': texture})\n    hu_table = pd.DataFrame({'Descriptor': [f'Log-transformed Hu {i+1}' for i in range(7)],\n                             'Actual feature value': hu})\n    return fig, texture_table, hu_table, feature\n",
  "app.py": "\"\"\"Local web demo; run after training. No model means no prediction.\"\"\"\nimport argparse\nfrom pathlib import Path\nimport cv2\nimport gradio as gr\nimport joblib\nimport numpy as np\nfrom pepper_model import VERSION, extract_features, preprocess, predict_all, display_label\n\nCSS = '''\n.gradio-container {max-width:1120px!important; margin:auto!important; background:var(--body-background-fill);}\n#hero {padding:34px 30px; background:#173c2c; border-radius:20px; margin-bottom:20px;}\n#hero h1,#hero p {color:#f4f7e9!important;}\n#hero h1 {font-size:42px; letter-spacing:-1.5px;}\n#hero p {max-width:700px; font-size:17px;}\n#analyze {background:#24543d!important; color:white!important; border-radius:12px;}\n'''\n\ndef default_model_path():\n    preferred = Path('artifacts_all_models/pepper_models.joblib')\n    return preferred if preferred.exists() else Path('artifacts/pepper_model.joblib')\n\ndef predict_image(image, bundle):\n    if image is None:\n        raise gr.Error('Drop a leaf photograph into the upload area first.')\n    if bundle is None:\n        raise gr.Error('No trained model found. Complete the training notebook first.')\n    try:\n        feature = extract_features(image)\n        rgb, _, hsv, mask, coverage = preprocess(image)\n        prediction = predict_all(bundle, feature)\n    except ValueError as exc:\n        raise gr.Error(str(exc)) from exc\n    overall = prediction['overall']\n    total = overall['total_models']\n    vote_shares = {display_label(c): n/total for c, n in overall['votes'].items()}\n    if overall.get('method') == 'validation_selected_model':\n        status = f\"### Main prediction: {overall['condition']}\\n\\n\"\n        status += f\"**Selected classifier: {overall['selected_model']}** \u2014 selected using validation data.\\n\\n\"\n        status += f\"Condition category: {overall['condition_type']}.\\n\\n\"\n    elif overall['prediction']:\n        status = f\"### Overall prediction: {overall['condition']}\\n\\n\"\n        status += f\"**{overall['votes_for_leading_class']} of {total} classifiers agree.** \"\n        status += f\"Condition category: {overall['condition_type']}.\\n\\n\"\n    else:\n        leaders = ', '.join(display_label(c) for c in overall['leading_classes'])\n        status = '### Overall prediction: Inconclusive\\n\\n'\n        status += f'No class received a majority. Leading class(es): {leaders}. Try a clearer photo.\\n\\n'\n    status += 'Vote shares show agreement between classifiers. Model scores are uncalibrated; neither is a diagnostic probability. '\n    status += 'The model can identify only the trained classes and does not verify crop identity.'\n    if not overall['seven_model_comparison']:\n        status += '\\n\\n**Legacy bundle:** only the available classifier is shown. Train the updated notebook to enable all seven.'\n    table = [[r['model'], r['condition'], r['condition_type'],\n              f\"{r['model_score']:.1%}\" if r['model_score'] is not None else 'Not available']\n             for r in prediction['per_model']]\n    segmented = cv2.bitwise_and(rgb, rgb, mask=mask)\n    gallery = [(rgb, 'RGB input'), (hsv[:, :, 0], 'Hue channel (0\u2013179)'),\n               (mask, 'Candidate leaf mask'), (segmented, 'Pixels used for colour features')]\n    details = f'Candidate mask covers {coverage:.1%} of the resized image. Check that it follows the leaf. This is not a disease-area measurement.'\n    return vote_shares, status, table, gallery, details, prediction\n\ndef build_app(model_path=None):\n    path = Path(model_path) if model_path else default_model_path()\n    bundle = joblib.load(path) if path.exists() else None\n    if bundle and bundle['feature_version'] != VERSION:\n        raise ValueError('Model and feature code do not match. Retrain with this version.')\n    with gr.Blocks(title='PepperLens | DSP leaf lab', analytics_enabled=False) as demo:\n        gr.Markdown('# PepperLens\\nA closer look at leaf health.\\n\\nDrop in a bell pepper leaf and explore what colour, texture, and shape reveal.', elem_id='hero')\n        labels = ', '.join(display_label(c) for c in bundle['classes']) if bundle else 'No model loaded'\n        count = len(bundle.get('models', {'Random Forest': None})) if bundle else 0\n        gr.Markdown(f'**{count} trained classifiers \u00b7 Trained conditions: {labels}**\\n\\nBest input: one leaf on a plain background.')\n        if bundle is None:\n            gr.Markdown('**Model not trained yet.** Run the notebook before making predictions.')\n        with gr.Row():\n            with gr.Column():\n                upload = gr.Image(label='Drop a leaf image here', type='numpy', image_mode='RGB', sources=['upload'], height=340)\n                button = gr.Button('Analyze leaf', variant='primary', elem_id='analyze')\n            with gr.Column():\n                result = gr.Label(label='Optional classifier vote comparison \u00b7 not the main decision', num_top_classes=len(bundle['classes']) if bundle else 2)\n                status = gr.Markdown('Your result will appear here.')\n        comparison = gr.Dataframe(headers=['Classifier', 'Predicted condition', 'Condition category', 'Model score (uncalibrated)'],\n                                   datatype=['str', 'str', 'str', 'str'], interactive=False,\n                                   label='Every classifier\u2019s prediction', wrap=True)\n        gr.Markdown('SVM supplies a class vote without a probability score. Scores from different algorithms are not directly comparable.')\n        with gr.Accordion('Explore the image processing', open=True):\n            gallery = gr.Gallery(label='Processing stages', columns=4, height=260)\n            details = gr.Markdown()\n        gr.Markdown('**How it works:** RGB \u2192 HSV colour mask \u2192 colour, texture and shape features \u2192 seven classifiers. The primary classifier is selected using validation balanced accuracy; all seven results are shown.\\n\\nA disease name is available only after training on that class. Nutrient deficiency is shown as an abiotic condition. Field performance has not been established.')\n        with gr.Accordion('Prediction details as JSON', open=False):\n            raw = gr.JSON(label='Per-classifier results and overall decision')\n        button.click(lambda image: predict_image(image, bundle), inputs=upload,\n                     outputs=[result, status, comparison, gallery, details, raw])\n    return demo\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--model', default=None)\n    parser.add_argument('--port', type=int, default=7860)\n    args = parser.parse_args()\n    build_app(args.model).launch(server_name='127.0.0.1', server_port=args.port, share=False, css=CSS,\n                                theme=gr.themes.Soft(primary_hue='green', neutral_hue='stone'))\n",
  "predict.py": "\"\"\"Predict user-supplied leaf photos with every fitted classifier and an overall vote.\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\nimport joblib\nfrom pepper_model import extract_features, read_rgb, predict_all\n\ndef predict_files(paths, model_path):\n    bundle = joblib.load(model_path)\n    results = []\n    for path in paths:\n        try:\n            result = predict_all(bundle, extract_features(read_rgb(path)))\n            results.append({'image': str(path), **result})\n        except (ValueError, OSError) as exc:\n            results.append({'image': str(path), 'error': str(exc)})\n    return results\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('images', nargs='+')\n    parser.add_argument('--model', default='artifacts_all_models/pepper_models.joblib')\n    parser.add_argument('--output', default=None)\n    args = parser.parse_args()\n    serialized = json.dumps(predict_files(args.images, args.model), indent=2, ensure_ascii=True)\n    if args.output:\n        Path(args.output).write_text(serialized, encoding='utf-8')\n    print(serialized)\n",
  "requirements.txt": "numpy>=1.26,<3\nopencv-python-headless>=4.9,<6\nmahotas>=1.4.13,<2\nscikit-learn>=1.4,<2\npandas>=2.1,<4\npillow>=10,<13\njoblib>=1.3,<2\ngradio>=6,<7\n\nipywidgets>=8,<9\nmatplotlib>=3.8,<4\nkagglehub\n"
}

from pathlib import Path
import hashlib
import importlib.util
import json
import os
import sys

BASE = (Path('/kaggle/working') if os.environ.get('KAGGLE_KERNEL_RUN_TYPE')
        else Path('/content') if Path('/content').is_dir() else Path.cwd())
CODE_ROOT = BASE / 'pepperlens_final_code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
for filename, source in SOURCES.items():
    target = CODE_ROOT / filename
    assert target.resolve().parent == CODE_ROOT.resolve()
    temporary = target.with_suffix(target.suffix + '.new')
    temporary.write_text(source, encoding='utf-8')
    temporary.replace(target)
source_hashes = {name: hashlib.sha256(source.encode('utf-8')).hexdigest() for name, source in SOURCES.items()}
(CODE_ROOT / 'source_versions.json').write_text(json.dumps(source_hashes, indent=2), encoding='utf-8')

def load_project_module(name):
    path = CODE_ROOT / (name + '.py')
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    # Compile the embedded source directly: no stale .pyc or previous repair cell.
    exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), module.__dict__)
    return module

cloud = load_project_module('cloud_data')
project = load_project_module('pepper_model')
fetch = load_project_module('fetch_mendeley')
notebook_ui = load_project_module('notebook_ui')
print('Embedded source and imports verified.')
print('Code directory:', CODE_ROOT)
print('Feature version:', project.VERSION)

## A. The DSP pipeline: visible implementations used by this program

An image is a sampled, quantized 2-D signal, $I[m,n,c]$. The learning algorithms operate **after** the signal-processing front end:

**Decode / orient → resize → Gaussian convolution → RGB to HSV → threshold masks → morphology / contour filling → colour + texture + shape descriptors → train seven classifiers.**

| Stage | DSP / image-processing concept | Role in this project |
|---|---|---|
| RGB / BGR | Channel-order representation | Prevents incorrect colour interpretation; not itself filtering. |
| Resize to 256 × 256 | Spatial resampling | Standard input grid and compute cost; interpolation changes pixel samples. |
| Gaussian smoothing | 2-D FIR convolution / spatial low-pass filtering | Reduces fine spatial intensity changes before segmentation and texture measurement. |
| RGB → HSV | Nonlinear colour-coordinate transformation | Separates hue, saturation and value for colour-based thresholds. |
| Masks, OR, closing, contour fill | Nonlinear segmentation and mathematical morphology | Produces a candidate leaf region including internal lesions. |
| Masked HSV histogram | Quantization / statistical feature extraction | 8 × 8 × 8 = 512 colour-distribution values. |
| Haralick descriptors | Gray-level co-occurrence / spatial statistics | 13 texture values from 32 gray levels, with zero reserved for background. |
| Hu moments | Normalized spatial moments | Seven log-transformed shape values from the binary silhouette. |
| DFT and Sobel plots below | Frequency-domain and gradient analysis | Educational diagnostics; not extra classifier inputs. |

The classifier receives **512 + 13 + 7 = 532 features**. Dataset auditing (bad files / duplicate labels) is different from image preprocessing (filtering / segmentation); both are retained and shown. No CNN, wavelet pipeline or bag-of-visual-words implementation is claimed.

### A1. Decode the image and handle RGB / BGR correctly

Pillow decodes directly to RGB and applies camera orientation. OpenCV's `imread` normally produces BGR; when using it, convert with `cv2.COLOR_BGR2RGB`. This program uses the Pillow path consistently, so it does not repeatedly swap channels. A later demonstration shows the incorrect display of a BGR array and the corrected RGB display.

The definitions below are executable, visible code. They are attached to the shared module so the training call uses them. The notebook generator extracts them from the packaged source to avoid maintaining two different implementations.

In [ ]:
import argparse
import hashlib
import json
import platform
from pathlib import Path
import cv2
import joblib
import mahotas
import numpy as np
import pandas as pd
import sklearn
from PIL import Image, ImageOps
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SIZE = project.SIZE
LABELS = project.LABELS
VERSION = project.VERSION

def read_rgb(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert('RGB'))

# Training uses this exact visible implementation.
project.read_rgb = read_rgb

### A2. Resampling, Gaussian convolution, HSV conversion and leaf segmentation

The Gaussian kernel has 3 × 3 taps with $\sigma=0.7$ pixels. The smoothing operation is a 2-D weighted neighborhood sum, $y[m,n]=\sum_i\sum_j h[i,j]x[m-i,n-j]$, with reflected borders in OpenCV. The implementation uses a finite sampled kernel; its measured frequency response is plotted later.

HSV thresholds select green and brown candidates. **Bitwise OR** combines masks without uint8 addition overflow. A 5 × 5 closing operation (dilation followed by erosion) closes small gaps. Filling the largest outer contour retains pale/dark internal spots. These thresholds are heuristic, and the selected region is **not** a disease lesion map or severity percentage. Nearly empty masks are rejected rather than silently classified.

In [ ]:
def preprocess(rgb):
    rgb = np.asarray(rgb)
    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:
        raise ValueError('Expected an RGB uint8 image with three channels.')
    rgb = cv2.resize(rgb, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    smooth = cv2.GaussianBlur(rgb, (3, 3), 0.7)
    hsv = cv2.cvtColor(smooth, cv2.COLOR_RGB2HSV)
    # Colour-based candidate leaf region. This is not a lesion annotation.
    green = cv2.inRange(hsv, (25, 35, 20), (100, 255, 255))
    brown = cv2.inRange(hsv, (5, 35, 15), (30, 255, 255))
    union = cv2.bitwise_or(green, brown)
    union = cv2.morphologyEx(union, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(union, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros(union.shape, np.uint8)
    if contours:
        # Filling the contour preserves dark / pale lesions inside the leaf.
        cv2.drawContours(mask, [max(contours, key=cv2.contourArea)], -1, 255, cv2.FILLED)
    coverage = float(np.count_nonzero(mask) / mask.size)
    if coverage < 0.01:
        raise ValueError('Too little usable leaf colour. Try a clear, close photo of one leaf.')
    return rgb, smooth, hsv, mask, coverage

# Training uses this exact visible implementation.
project.preprocess = preprocess

### A3. Convert the processed image into the actual 532 classifier inputs

The HSV histogram uses the leaf mask, including OpenCV's correct hue range **0–179**. Haralick texture uses quantized grayscale inside that region and ignores zero background. Hu features use binary shape moments with a signed log transform. The same extractor is used for training and new-photo prediction; background pixels are not treated as leaf colour.

In [ ]:
def extract_features(rgb):
    rgb, smooth, hsv, mask, coverage = preprocess(rgb)
    histogram = cv2.calcHist([hsv], [0, 1, 2], mask, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    histogram = histogram.ravel() / max(float(histogram.sum()), 1.0)
    gray = cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY)
    # Reserve zero for background; use 32 gray levels to keep GLCMs cheap.
    quantized = np.where(mask > 0, gray // 8 + 1, 0).astype(np.uint8)
    texture = mahotas.features.haralick(quantized, ignore_zeros=True, return_mean=True)
    moments = cv2.HuMoments(cv2.moments(mask, binaryImage=True)).ravel()
    moments = -np.sign(moments) * np.log10(np.maximum(np.abs(moments), 1e-30))
    feature = np.concatenate([histogram, texture, moments]).astype(np.float32)
    if feature.shape != (532,) or not np.isfinite(feature).all():
        raise ValueError('Invalid feature vector; inspect the input image.')
    return feature

# Training uses this exact visible implementation.
project.extract_features = extract_features

## B. Dataset cleaning, grouping and compact feature storage

### B1. Inspect every file and audit incorrect labels / duplicates

The next visible function decodes files, records unreadable images, hashes decoded RGB pixels, reports conflicting labels and excludes **all copies** of conflicting exact images. It then retains one copy of each unambiguous exact image. No source photograph is renamed, deleted or assigned a guessed label. Available parent/plant IDs and detected duplicate links form connected groups before exclusions, so removed images do not break those connections.

The DCT-based perceptual hash here is a **duplicate-grouping heuristic**, not a classification feature or proof that every transformed sibling has been found.

In [ ]:
def make_manifest(root, output, groups_csv=None, class_names=None, conflict_policy='exclude'):
    """Index files without copying, renaming, or deleting any source image.

    groups_csv: path,group_id columns; paths relative to root or absolute.
    Group IDs must identify the same plant/source image across derivatives.
    Conflicting decoded duplicates are reported and all copies are excluded by
    default. 'error' writes the same audit, then stops for manual review.
    """
    root, output = Path(root).resolve(), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    if conflict_policy not in {'exclude', 'error'}:
        raise ValueError("conflict_policy must be 'exclude' or 'error'.")
    manual = {}
    if groups_csv:
        for r in pd.read_csv(groups_csv, dtype=str).itertuples():
            manual[str((root / r.path).resolve())] = r.group_id
    leaf_map_file = root / 'leaf-map.json'
    leaf_map = json.loads(leaf_map_file.read_text()) if leaf_map_file.exists() else {}
    upstream_classes = {'bacterial_spot': 'Pepper,_bell___Bacterial_spot',
                        'healthy': 'Pepper,_bell___healthy'}
    class_names = list(class_names) if class_names is not None else LABELS
    if len(class_names) < 2 or len(set(class_names)) != len(class_names):
        raise ValueError('Provide at least two distinct class names.')
    rows, rejected = [], []
    for label in class_names:
        folder = root / label
        if not folder.is_dir():
            raise ValueError(f'Missing class folder: {folder}')
        for path in sorted(folder.rglob('*')):
            if path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
                continue
            try:
                rgb = read_rgb(path)
                digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
                tiny = cv2.resize(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (32, 32))
                dct = cv2.dct(tiny.astype(np.float32))[:8, :8].ravel()[1:]
                phash = ''.join('1' if v > np.median(dct) else '0' for v in dct)
                identifier = path.stem.replace('_final_masked', '').split('___')[-1].split('copy')[0].strip().lower()
                suggestions = leaf_map.get(identifier, [])
                upstream_class = upstream_classes.get(label)
                matches = [s for s in suggestions if upstream_class and upstream_class in s]
                leaf_id = suggestions[0] if len(suggestions) == 1 else (matches[0] if len(matches) == 1 else None)
                source_group = manual.get(str(path), leaf_id or digest)
                basis = 'manual' if str(path) in manual else ('upstream_leaf_id' if leaf_id else 'image_hash_only')
                rows.append({'path': str(path), 'label': label, 'sha256': digest,
                             'phash': phash, 'group_id': source_group,
                             'source_group': source_group, 'group_basis': basis})
            except Exception as exc:
                rejected.append({'path': str(path), 'reason': str(exc)})
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError('No readable images found.')
    missing = set(class_names) - set(df.label)
    if missing:
        raise ValueError(f'No readable images in requested class(es): {sorted(missing)}')
    label_counts = df.groupby('sha256').label.nunique()
    conflict_hashes = set(label_counts[label_counts > 1].index)
    conflicts = df[df.sha256.isin(conflict_hashes)].copy()
    label_sets = df.groupby('sha256').label.agg(lambda values: ' | '.join(sorted(set(values))))
    conflicts['conflicting_labels'] = conflicts.sha256.map(label_sets)
    conflicts['action'] = 'excluded_from_training_and_evaluation' if conflict_policy == 'exclude' else 'requires_manual_review'
    conflicts.to_csv(output / 'label_conflicts.csv', index=False)
    pd.DataFrame(rejected, columns=['path', 'reason']).to_csv(output / 'unreadable.csv', index=False)
    df[df.duplicated('sha256', keep=False)].to_csv(output / 'exact_duplicates.csv', index=False)
    counts_before = {label: int(df.label.eq(label).sum()) for label in class_names}
    audit = {
        'conflict_policy': conflict_policy,
        'readable_files_before_cleanup': len(df), 'unreadable_files': len(rejected),
        'conflicting_image_groups': len(conflict_hashes), 'conflicting_files_detected': len(conflicts),
        'conflicting_files_excluded': len(conflicts) if conflict_policy == 'exclude' else 0,
        'counts_before_cleanup': counts_before,
    }
    if conflict_hashes and conflict_policy == 'error':
        audit['status'] = 'blocked_for_label_review'
        (output / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
        raise ValueError(f'Identical decoded images have conflicting labels: {len(conflicts)} files '
                         f'in {len(conflict_hashes)} groups. See {output / "label_conflicts.csv"}. '
                         "Use conflict_policy='exclude' to omit every copy without changing source files.")
    # Connect matching metadata groups, exact images, and exact perceptual hashes.
    # This is a conservative heuristic, NOT a complete near-duplicate detector.
    parent = list(range(len(df)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for column in ['group_id', 'sha256', 'phash']:
        first = {}
        for i, key in enumerate(df[column]):
            if key in first:
                parent[find(i)] = find(first[key])
            else:
                first[key] = i
    df['group_id'] = [f'group-{find(i)}' for i in range(len(df))]
    # Form groups BEFORE exclusion/deduplication so removed rows cannot break a
    # connection between surviving parent groups or transformed siblings.
    unambiguous = df[~df.sha256.isin(conflict_hashes)].copy()
    df = unambiguous.drop_duplicates('sha256').reset_index(drop=True)
    audit['same_label_duplicate_copies_removed'] = len(unambiguous) - len(df)
    audit['retained_unique_images'] = len(df)
    audit['counts_after_cleanup'] = {label: int(df.label.eq(label).sum()) for label in class_names}
    missing_after_cleanup = set(class_names) - set(df.label)
    audit['status'] = 'blocked_missing_classes_after_cleanup' if missing_after_cleanup else 'ready'
    (output / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    df.to_csv(output / 'manifest.csv', index=False)
    if missing_after_cleanup:
        raise ValueError('No usable images remain after label-conflict exclusion in class(es): '
                         f'{sorted(missing_after_cleanup)}. Review {output / "label_conflicts.csv"}; '
                         'add correctly labelled examples before training.')
    if conflict_hashes:
        print(f'[AUDIT] Excluded all {len(conflicts)} files in {len(conflict_hashes)} conflicting-label '
              'groups. Source images were preserved. See label_conflicts.csv in the output folder.', flush=True)
    print(f'[AUDIT] Retained {len(df)} unique unambiguous images. '
          f'Per class: {audit["counts_after_cleanup"]}', flush=True)
    df.attrs['data_audit'] = audit
    return df

# Training uses this exact visible implementation.
project.make_manifest = make_manifest

### B2. Keep detected groups together across train, validation and test

Five stratified grouped folds produce an approximate 60% / 20% / 20% division. One fold is test, one is validation and three are training. This is a single grouped holdout experiment, not a five-fold cross-validation score. Every partition must contain every requested class; the code does not silently discard a class. Model choice uses validation only.

In [ ]:
def assign_splits(df, seed=42):
    df = df.copy()
    labels = set(df.label)
    if df.groupby('label').group_id.nunique().min() < 5:
        raise ValueError('Need at least five independent groups per class.')
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    df['split'] = 'train'
    for fold, (_, indices) in enumerate(folds.split(df, df.label, df.group_id)):
        if fold in (0, 1):
            df.loc[df.index[indices], 'split'] = 'test' if fold == 0 else 'validation'
    if df.groupby('group_id').split.nunique().max() != 1:
        raise AssertionError('A group crosses partitions.')
    for split in ['train', 'validation', 'test']:
        if set(df.loc[df.split == split, 'label']) != labels:
            raise ValueError(f'{split} lacks one class. Review group sizes before changing the seed.')
    return df

# Training uses this exact visible implementation.
project.assign_splits = assign_splits

### B3. Read images individually and cache numerical feature vectors

Thousands of photographs are not loaded into a single RAM array. Images are processed one by one into 532 float32 values, cached by decoded-image hash and stored in a disk-backed matrix. Corrupt/incomplete cache entries are recomputed. Feature-extraction failures are listed separately in `feature_failures.csv`; the final accuracy also counts test rejections as errors.

In [ ]:
def cached_features(df, output):
    output = Path(output)
    cache = output / 'feature_cache' / VERSION
    cache.mkdir(parents=True, exist_ok=True)
    matrix = np.lib.format.open_memmap(output / 'features.npy', mode='w+', dtype='float32', shape=(len(df), 532))
    failures, valid = [], np.ones(len(df), dtype=bool)
    for i, r in enumerate(df.itertuples()):
        file = cache / (r.sha256 + '.npy')
        try:
            feature = None
            if file.exists():
                try:
                    candidate = np.load(file, allow_pickle=False)
                    if candidate.shape == (532,) and np.isfinite(candidate).all():
                        feature = candidate
                except (OSError, ValueError, EOFError):
                    pass  # interrupted/corrupt cache: recover from the unchanged source image
            if feature is None:
                feature = extract_features(read_rgb(r.path))
                temp = file.with_suffix('.tmp')
                with temp.open('wb') as handle:
                    np.save(handle, feature)
                temp.replace(file)
            matrix[i] = feature
        except Exception as exc:
            valid[i] = False
            matrix[i] = 0
            failures.append({'path': r.path, 'label': r.label, 'split': r.split, 'reason': str(exc)})
        if (i + 1) % 250 == 0:
            print(f'Features {i+1}/{len(df)}', flush=True)
    matrix.flush()
    pd.DataFrame(failures, columns=['path', 'label', 'split', 'reason']).to_csv(output / 'feature_failures.csv', index=False)
    return matrix, valid

# Training uses this exact visible implementation.
project.cached_features = cached_features

## C. The ML stage after digital image processing

These seven algorithms receive the same DSP descriptors. Scaling is fitted inside training-only pipelines where needed. The model with the best **validation balanced accuracy** becomes the main predictor before evaluating test rows. Random Forest is eligible to win but is not forced to be the winner. The default has no added synthetic training views; the larger dataset's own augmentation history remains a provenance limitation.

In [ ]:
def make_models(seed=42):
    """All seven original algorithm families; scaling is fitted on training only.

    LDA uses shrinkage to handle correlated/constant histogram columns. SVM
    deliberately has no internally fitted probability calibration; it votes using
    predict(), so ungrouped calibration folds cannot leak leaf identities.
    """
    return {
        'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(
            max_iter=3000, class_weight='balanced', random_state=seed)),
        'Linear Discriminant Analysis': make_pipeline(StandardScaler(),
            LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
        'K-Nearest Neighbors': make_pipeline(StandardScaler(),
            KNeighborsClassifier(n_neighbors=7, weights='distance', algorithm='ball_tree', n_jobs=2)),
        'Decision Tree': DecisionTreeClassifier(min_samples_leaf=2,
            class_weight='balanced', random_state=seed),
        'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12,
            min_samples_leaf=2, class_weight='balanced', n_jobs=2, random_state=seed),
        'Gaussian Naive Bayes': make_pipeline(StandardScaler(), GaussianNB(var_smoothing=1e-8)),
        'Support Vector Machine': make_pipeline(StandardScaler(),
            SVC(C=5, class_weight='balanced', random_state=seed)),
    }

# Training uses this exact visible implementation.
project.make_models = make_models

In [ ]:
# Refresh the prediction helper so it references the visible extractor above.
notebook_ui = load_project_module('notebook_ui')
assert project.preprocess is preprocess and project.extract_features is extract_features
print('Visible image-processing, cleaning and classifier definitions are connected to training.')

**Automatic data access.** The default Mendeley archive API is embedded below and needs no API token. It downloads a single archive and extracts class folders automatically. No manual training-image insertion is needed. The alternative requests the Bell pepper directory through KaggleHub; Kaggle may require login/consent for that service and it supplies only healthy/bacterial-spot labels.

The larger source's original/augmented counts and parent IDs are not established. All detected conflicts and duplicates are audited before splitting; unseen transformed copies may still exist. Keep a separate independently collected final field test.

In [ ]:
# 3. Automatic dataset access. Defaults need no edits for the six-class task.
import json
import shutil

DATA_SOURCE = 'mendeley_bell_6'  # alternative: 'plantifydr_pepper'
GROUPS_CSV = None               # optional reviewed parent/plant group metadata
TARGET_TRAIN_VIEWS = 0          # use the actual data first; no synthetic size inflation
OUTPUT = BASE / 'pepper_all_models'
DATASET_PAGE = 'https://data.mendeley.com/datasets/w9drvtxg9g/1'
DATASET_API = 'https://data.mendeley.com/public-api/zip/w9drvtxg9g/download/1'

cloud.require_cloud_runtime()  # intentionally prevents new dataset downloads on your PC
print('Source:', DATASET_PAGE)
print('Free cloud disk:', round(shutil.disk_usage(BASE).free / 1e9, 2), 'GB')
if DATA_SOURCE == 'mendeley_bell_6':
    DATA_ROOT = BASE / 'pepper_data'
    CLASS_NAMES = list(project.MULTICLASS_LABELS)
    archive = BASE / 'mendeley_pepper_v1.zip'
    reusable = False
    source_file = DATA_ROOT / 'source.json'
    if source_file.is_file():
        try:
            previous_source = json.loads(source_file.read_text(encoding='utf-8'))
            expected = previous_source.get('measured_file_counts', {})
            reusable = (previous_source.get('source') == DATASET_PAGE and set(expected) == set(CLASS_NAMES)
                        and all(expected[label] > 0 and sum(p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
                            for p in (DATA_ROOT / label).rglob('*')) == expected[label] for label in CLASS_NAMES))
        except (OSError, ValueError, KeyError):
            reusable = False
    if reusable:
        print('Reusing the complete dataset already extracted in this cloud session.')
    else:
        fetch.URL = DATASET_API
        DATA_ROOT = fetch.download(DATA_ROOT, archive, task='multiclass')
elif DATA_SOURCE == 'plantifydr_pepper':
    DATA_ROOT = cloud.import_plantifydr(BASE / 'pepper_plantifydr')
    CLASS_NAMES = ['bacterial_spot', 'healthy']
else:
    raise ValueError('Choose one of the documented API sources.')
print('Training classes:', CLASS_NAMES)
if GROUPS_CSV is None:
    print('Evaluation is provisional: augmentation/plant parent IDs are not reviewed. Hashes cannot prove independence.')

**Train → validate → select → test.** Approximately 60% of independent detected groups are used for training, 20% for validation and 20% for testing. Models/scalers are fitted using training rows only. The highest validation balanced accuracy selects the primary classifier; ties use the declared model order. That choice is frozen before the test set is evaluated. The test score is never used to choose the final classifier.

Conflicting identical images are all omitted, with filenames retained in `label_conflicts.csv`. `data_audit.json` reports counts before and after cleaning. A class becoming empty or lacking sufficient groups remains a real data error rather than being silently dropped. Feature extraction reads images individually and caches compact feature vectors so completed work can be reused on reruns.

In [ ]:
# 4. Train all seven classifiers and select the main one using validation only.
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'run_status.json').write_text(json.dumps({'status': 'training', 'source': DATA_SOURCE}), encoding='utf-8')
try:
    report = project.train(
        DATA_ROOT, OUTPUT, GROUPS_CSV, TARGET_TRAIN_VIEWS,
        class_names=CLASS_NAMES, conflict_policy='exclude',
        data_note=f'{DATA_SOURCE}; conflicting exact duplicates excluded; external field evaluation pending',
    )
except Exception as error:
    (OUTPUT / 'run_status.json').write_text(json.dumps({'status': 'failed', 'error': str(error)}), encoding='utf-8')
    raise
else:
    (OUTPUT / 'run_status.json').write_text(json.dumps({'status': 'complete', 'selected_model': report['selected_model']}), encoding='utf-8')
print('MAIN CLASSIFIER:', report['selected_model'])
print('Validation balanced accuracy:', f"{report['selection_score']:.2%}")
print('Final test accuracy, including feature rejections as errors:',
      f"{report['test_accuracy_counting_rejections_as_errors']:.2%}")

In [ ]:
# 5. Inspect the actual cleaned counts, model comparison and selected-model results.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

audit = report['data_audit']
display(pd.DataFrame({'Before cleanup': audit['counts_before_cleanup'], 'Retained unique images': audit['counts_after_cleanup']}))
print('Conflicting files excluded:', audit['conflicting_files_excluded'])
print('Same-label duplicate copies removed:', audit['same_label_duplicate_copies_removed'])
display(pd.read_csv(OUTPUT / 'label_conflicts.csv').head(20))
manifest = pd.read_csv(OUTPUT / 'manifest.csv')
display(pd.crosstab(manifest['split'], manifest['label']))
comparison = pd.read_csv(OUTPUT / 'model_comparison.csv')
percentage_table = comparison.copy()
for column in ['validation_balanced_accuracy', 'validation_macro_f1', 'test_accuracy', 'test_balanced_accuracy', 'test_macro_f1']:
    percentage_table[column] = (100 * percentage_table[column]).round(2)
print('Scores below are percentages. Main classifier was selected on validation balanced accuracy.')
display(percentage_table)
display(pd.DataFrame(report['classification_report']).T)
print('Feature coverage:', f"{report['test_coverage']:.2%}")
print('Optional voting comparison:', f"{report['ensemble_accuracy_counting_rejections_and_abstentions_as_errors']:.2%}")
print('These are internal results on the cleaned subset; field accuracy remains unmeasured.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
positions = np.arange(len(comparison))
axes[0].bar(positions - .18, 100*comparison.validation_balanced_accuracy, .36, label='Validation balanced accuracy')
axes[0].bar(positions + .18, 100*comparison.test_balanced_accuracy, .36, label='Test balanced accuracy')
axes[0].set_xticks(positions, ['LR', 'LDA', 'KNN', 'DT', 'RF', 'GNB', 'SVM'])
axes[0].set_ylim(0, 105)
axes[0].set_ylabel('Percent')
axes[0].set_title('Same partitions, all seven classifiers')
axes[0].legend(fontsize=8)
labels = report['labels']
cm = np.asarray(report['confusion_matrix'])[:len(labels), :len(labels)]
axes[1].imshow(cm, cmap='Greens')
axes[1].set_xticks(range(len(labels)), [project.display_label(c) for c in labels], rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(len(labels)), [project.display_label(c) for c in labels], fontsize=8)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Selected classifier: ' + report['selected_model'])
for i in range(len(labels)):
    for j in range(len(labels)):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / 'model_results.png', dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

## D. DSP laboratory: see what happens to a real training image

These plots use a usable example from the **training partition**. They are illustrative demonstrations, not diagnostic evidence of disease or a way to choose parameters using test images. The actual clean counts are shown above, and the actual intermediate arrays are computed below. Raw source images remain unchanged.

In [ ]:
# Select training examples only; do not inspect the test holdout to tune processing.
training_rows = manifest[manifest['split'] == 'train']
demo_rgb = None
for row in training_rows.sort_values(['label', 'path']).itertuples():
    try:
        candidate = read_rgb(row.path)
        extract_features(candidate)
    except (OSError, ValueError):
        continue
    demo_rgb, demo_path, demo_class = candidate, row.path, row.label
    break
if demo_rgb is None:
    raise ValueError('No usable training example for the DSP demonstration.')
print('DSP demonstration source:', demo_path)
print('Dataset label (not a new prediction):', project.display_label(demo_class))
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(3*len(CLASS_NAMES), 3), squeeze=False)
for axis, label in zip(axes.flat, CLASS_NAMES):
    first = training_rows[training_rows.label == label].iloc[0]
    axis.imshow(read_rgb(first.path))
    axis.set_title(project.display_label(label), fontsize=9)
    axis.axis('off')
fig.suptitle('One retained training example per class')
fig.tight_layout()
display(fig)
fig.savefig(OUTPUT / 'training_class_examples.png', dpi=140, bbox_inches='tight')
plt.close(fig)

### D1. RGB / BGR / HSV, filtering, thresholding and morphology: 16 intermediate views

The BGR panel is deliberately displayed incorrectly to demonstrate channel-order errors; the next panel restores RGB. HSV is displayed as separate channels, because showing the three-channel HSV array as an RGB photograph gives misleading colours. Sobel and DFT panels illustrate spatial derivatives and spatial-frequency content; the classifiers still use only the original 532 descriptors.

In [ ]:
def plot_processing_stages(image):
    """Show channel order, colour coordinates, nonlinear masks and segmentation."""
    rgb, smooth, hsv, mask, coverage = preprocess(image)
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    restored = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    np.testing.assert_array_equal(restored, rgb)
    green = cv2.inRange(hsv, (25, 35, 20), (100, 255, 255))
    brown = cv2.inRange(hsv, (5, 35, 15), (30, 255, 255))
    union = cv2.bitwise_or(green, brown)
    closed = cv2.morphologyEx(union, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    segmented = rgb.copy()
    segmented[mask == 0] = 0
    gray = cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    edge_magnitude = np.hypot(gx, gy)
    log_spectrum = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray))))
    stages = [
        (rgb, '1. Resampled RGB: 256 x 256', None, None),
        (bgr, '2. BGR incorrectly shown as RGB\n(channel-order demonstration)', None, None),
        (restored, '3. Correct BGR -> RGB display', None, None),
        (smooth, '4. Gaussian smoothing\n3 x 3, sigma = 0.7 pixels', None, None),
        (hsv[:, :, 0], '5. Hue H: OpenCV range 0-179', 'hsv', (0, 179)),
        (hsv[:, :, 1], '6. Saturation S: 0-255', 'gray', (0, 255)),
        (hsv[:, :, 2], '7. Value V: 0-255', 'gray', (0, 255)),
        (gray, '8. Grayscale for texture', 'gray', (0, 255)),
        (green, '9. Green threshold mask', 'gray', (0, 255)),
        (brown, '10. Brown threshold mask', 'gray', (0, 255)),
        (union, '11. Mask union: bitwise OR', 'gray', (0, 255)),
        (closed, '12. Morphological closing', 'gray', (0, 255)),
        (mask, '13. Filled largest contour', 'gray', (0, 255)),
        (segmented, '14. Selected leaf pixels', None, None),
        (edge_magnitude, '15. Sobel edge magnitude\n(diagnostic only)', 'magma', None),
        (log_spectrum, '16. Log magnitude of 2-D DFT\n(diagnostic only)', 'magma', None),
    ]
    fig, axes = plt.subplots(4, 4, figsize=(15, 15))
    for axis, (pixels, title, colour_map, limits) in zip(axes.flat, stages):
        options = {'cmap': colour_map} if colour_map else {}
        if limits:
            options.update(vmin=limits[0], vmax=limits[1])
        axis.imshow(pixels, **options)
        axis.set_title(title, fontsize=10)
        axis.axis('off')
    fig.suptitle(f'Actual leaf-processing pipeline; candidate mask coverage {coverage:.1%}', fontsize=15)
    fig.tight_layout(rect=(0, 0, 1, .97))
    arrays = {'rgb': rgb, 'bgr': bgr, 'smooth': smooth, 'hsv': hsv, 'mask': mask,
              'green': green, 'brown': brown, 'closed': closed, 'segmented': segmented,
              'edge_magnitude': edge_magnitude, 'log_spectrum': log_spectrum}
    return fig, arrays

stage_figure, dsp_arrays = plot_processing_stages(demo_rgb)
display(stage_figure)
stage_figure.savefig(OUTPUT / 'dsp_processing_stages.png', dpi=150, bbox_inches='tight')
plt.close(stage_figure)

### D2. Show the actual Gaussian filter in the spatial and frequency domains

For the 3 × 3 sampled kernel, $h[i,j] = g[i]g[j]$ with $\sum h=1$. The notebook computes its 2-D DFT magnitude $|H(f_x,f_y)|$: DC has unit gain and higher spatial frequencies are attenuated. This connects the smoothing already used by the classifier to a DSP low-pass filter. Frequency is measured in **cycles per pixel**, with Nyquist at 0.5; these are not temporal frequencies in Hz.

The before/after image spectra use one display scale. Image boundary handling and 8-bit rounding mean these finite-image spectra need not equal simple pointwise multiplication by the plotted ideal periodic response. Sobel and Fourier results are demonstrations, not extra learned features or an accuracy improvement claim.

In [ ]:
def plot_filter_analysis(image):
    """Relate the Gaussian convolution used by the model to its frequency response."""
    rgb, smooth, _, _, _ = preprocess(image)
    gray_before = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)
    gray_after = cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY).astype(np.float64)
    gaussian_1d = cv2.getGaussianKernel(3, .7)
    kernel = gaussian_1d @ gaussian_1d.T
    response = np.abs(np.fft.fftshift(np.fft.fft2(kernel, s=gray_before.shape)))
    frequency = np.fft.fftshift(np.fft.fftfreq(gray_before.shape[1]))
    before_spectrum = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray_before))))
    after_spectrum = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray_after))))
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes[0, 0].imshow(kernel, cmap='Greens')
    for i in range(3):
        for j in range(3):
            axes[0, 0].text(j, i, f'{kernel[i, j]:.3f}', ha='center', va='center')
    axes[0, 0].set_title('Normalized Gaussian convolution kernel')
    axes[0, 1].imshow(response, cmap='viridis', extent=(-.5, .5, -.5, .5), vmin=0, vmax=1)
    axes[0, 1].set_title('Magnitude frequency response |H(fx, fy)|')
    axes[0, 1].set_xlabel('Horizontal cycles / pixel')
    axes[0, 1].set_ylabel('Vertical cycles / pixel')
    axes[0, 2].plot(frequency, response[response.shape[0]//2])
    axes[0, 2].set_title('Filter response along fy = 0')
    axes[0, 2].set_xlabel('Horizontal cycles / pixel')
    axes[0, 2].set_ylabel('Amplitude gain')
    axes[0, 2].set_ylim(0, 1.05)
    axes[0, 2].grid(alpha=.3)
    shared_max = max(before_spectrum.max(), after_spectrum.max())
    axes[1, 0].imshow(before_spectrum, cmap='magma', vmin=0, vmax=shared_max)
    axes[1, 0].set_title('Log DFT magnitude before smoothing')
    axes[1, 1].imshow(after_spectrum, cmap='magma', vmin=0, vmax=shared_max)
    axes[1, 1].set_title('Log DFT magnitude after smoothing\n(same display scale)')
    residual = gray_before - gray_after
    limit = max(float(np.abs(residual).max()), 1)
    axes[1, 2].imshow(residual, cmap='coolwarm', vmin=-limit, vmax=limit)
    axes[1, 2].set_title('Removed spatial detail: before - after')
    for axis in [axes[0, 0], *axes[1]]:
        axis.axis('off')
    fig.suptitle('DSP application: spatial low-pass filtering and its 2-D frequency response')
    fig.tight_layout(rect=(0, 0, 1, .95))
    return fig, kernel, response

filter_figure, gaussian_kernel, gaussian_response = plot_filter_analysis(demo_rgb)
display(filter_figure)
display(pd.DataFrame(gaussian_kernel))
filter_figure.savefig(OUTPUT / 'dsp_gaussian_frequency_response.png', dpi=160, bbox_inches='tight')
plt.close(filter_figure)

### D3. Inspect the exact colour, texture and shape values used by the ML stage

The three plotted histograms are marginals of the **same joint 8 × 8 × 8 histogram** supplied to the classifiers. They are not three replacement 256-bin histograms. The tables show the actual 13 Haralick and seven log-transformed Hu values. Different descriptor families have different scales; scaling belongs inside the classifier pipeline, fitted on training rows only.

In [ ]:
def plot_feature_summary(image):
    """Visualize the same 532 values supplied to every classifier."""
    feature = extract_features(image)
    histogram = feature[:512].reshape(8, 8, 8)
    texture = feature[512:525]
    hu = feature[525:532]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for axis, marginal, title, upper in zip(
            axes, [histogram.sum(axis=(1, 2)), histogram.sum(axis=(0, 2)), histogram.sum(axis=(0, 1))],
            ['Hue marginal', 'Saturation marginal', 'Value marginal'], [180, 256, 256]):
        centers = (np.arange(8) + .5) * upper / 8
        axis.bar(centers, marginal, width=upper/10, color='#36734b')
        axis.set_title(title + ' of masked 8 x 8 x 8 histogram')
        axis.set_xlabel('Channel-bin center')
        axis.set_ylabel('Fraction of selected pixels')
        axis.set_xlim(0, upper)
    fig.tight_layout()
    texture_table = pd.DataFrame({'Descriptor': [f'Haralick {i+1}' for i in range(13)],
                                  'Actual feature value': texture})
    hu_table = pd.DataFrame({'Descriptor': [f'Log-transformed Hu {i+1}' for i in range(7)],
                             'Actual feature value': hu})
    return fig, texture_table, hu_table, feature

feature_figure, haralick_table, hu_table, demo_features = plot_feature_summary(demo_rgb)
display(feature_figure)
display(haralick_table)
display(hu_table)
print('Feature vector:', demo_features.shape, demo_features.dtype)
print('Masked histogram sum:', float(demo_features[:512].sum()))
assert demo_features.shape == (532,) and np.isfinite(demo_features).all()
np.testing.assert_allclose(demo_features, project.extract_features(demo_rgb))
feature_figure.savefig(OUTPUT / 'dsp_colour_histograms.png', dpi=160, bbox_inches='tight')
haralick_table.to_csv(OUTPUT / 'dsp_haralick_example.csv', index=False)
hu_table.to_csv(OUTPUT / 'dsp_hu_example.csv', index=False)
plt.close(feature_figure)

In [ ]:
# 6. Save the compact result bundle before the cloud runtime expires.
import zipfile
bundle_path = BASE / 'PepperLens_FINAL_results.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT.iterdir():
        if path.is_file() and path.suffix in {'.json', '.csv', '.joblib', '.png'}:
            archive.write(path, 'artifacts_all_models/' + path.name)
    for path in CODE_ROOT.iterdir():
        if path.is_file() and path.suffix in {'.py', '.txt', '.json'}:
            archive.write(path, path.name)
    if (DATA_ROOT / 'source.json').is_file():
        archive.write(DATA_ROOT / 'source.json', 'dataset_source.json')
print('Saved:', bundle_path)
print('The ZIP contains models, source and reports; it excludes training images and feature caches.')
try:
    from google.colab import files as colab_files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(bundle_path)))
else:
    colab_files.download(str(bundle_path))

**Predict your photographs.** The notebook-native panel below does not require a web server or a public sharing link. Upload one prediction image, then click **Analyze leaf**. It shows the validation-selected result and all seven classifier predictions, plus RGB/HSV/mask stages. The training dataset was fetched automatically; only your own new prediction photographs are uploaded through this panel.

Only trained labels can be predicted. Nutrient deficiency is displayed as an abiotic condition, and leaf curl is the dataset's symptom class. Scores are uncalibrated. A confident known-class output does not prove the photograph is bell pepper or rule out an unseen disease.

In [ ]:
# 7. Server-free interactive prediction panel.
import joblib
from IPython.display import display
bundle = joblib.load(OUTPUT / 'pepper_models.joblib')
try:
    from google.colab import output as colab_output
except ImportError:
    pass
else:
    colab_output.enable_custom_widget_manager()
prediction_panel = notebook_ui.build_notebook_ui(bundle, OUTPUT)
display(prediction_panel)

def predict_my_images():
    # Native Colab alternative if your browser cannot render notebook widgets.
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        table, result, stages = notebook_ui.prediction_for_bytes(content, bundle)
        print(name, '→', result['overall']['condition'], 'using', result['overall']['selected_model'])
        display(table)
        display(stages)
print('If the upload panel is unavailable, run predict_my_images() in a new cell.')

**Optional website-style demo.** This is separate from the working notebook prediction panel. Change the switch below only if you want the Gradio interface. On Colab, Gradio may need a temporary **public sharing URL** to embed its interface. The source disables analytics. If that external service or the server cannot start, the notebook-native predictor remains available.

In [ ]:
# 8. Optional browser demo; the main workflow works without it.
START_WEB_DEMO = False
if START_WEB_DEMO:
    import gradio as gr
    web_app = load_project_module('app')
    public_share = os.environ.get('COLAB_RELEASE_TAG') is not None or Path('/content').is_dir()
    try:
        demo = web_app.build_app(OUTPUT / 'pepper_models.joblib')
        demo.launch(inline=True, share=public_share, prevent_thread_lock=True,
                    css=web_app.CSS, theme=gr.themes.Soft(primary_hue='green', neutral_hue='stone'))
    except Exception as error:
        print('Web demo could not start:', error)
        print('Use the notebook-native prediction panel above; the trained models are already saved.')
else:
    print('Notebook prediction panel is ready. Optional web-server launch is disabled.')

## DSP concepts and experimental limits

Photographs are sampled 2-D signals. The visible shared extractor uses resampling, Gaussian smoothing (2-D convolution), RGB→HSV colour transformation, thresholding and morphology, a 512-value masked colour histogram, 13 Haralick co-occurrence texture descriptors and seven Hu spatial moments. The classifier sees the same 532 features at training and prediction time. Random Forest is one candidate; all seven families remain available for comparison. DCT is used only for duplicate-audit hashing. DFT / Sobel are educational diagnostics, not classification features. No CNN or bag-of-visual-words method is implemented.

Suggested viva explanation: “We treat leaf photographs as sampled two-dimensional signals. A Gaussian low-pass filter smooths them; HSV thresholds and morphology select a candidate leaf region. We measure colour distributions, co-occurrence texture and normalized spatial moments. Seven machine-learning models classify those descriptors. We select the main classifier on validation data and evaluate it on a separate grouped test split. We demonstrate the filter in both the spatial and frequency domains.”

Keep source/augmentation-parent groups together. Do not retune the method using the final test results and then claim the same test is untouched. After further development, use a fresh independently collected field holdout. The published 9,283-file count is not a promise of 9,283 independent clean leaves. The audit's retained counts and exclusions must accompany any accuracy in your coursework.

Sources: [bell pepper dataset and license](https://data.mendeley.com/datasets/w9drvtxg9g/1), [KaggleHub API](https://github.com/Kaggle/kagglehub), [scikit-learn model evaluation](https://scikit-learn.org/stable/model_selection.html), [Gradio sharing behavior](https://gradio.app/guides/sharing-your-app).